# Google ADK 2.0 Graph API: Hands-On Tutorial (Basic to Intermediate)

**Model used throughout:** `gemini-3.5-flash` (Gemini 3.5 Flash)
**Framework:** Google Agent Development Kit (ADK) for Python, version 2.x

This notebook teaches you how to build AI applications with the ADK 2.0 **Graph API** (the `Workflow` runtime). Every part follows the same rhythm:

1. **Concepts first**: what the idea is, why it exists, and when to use it.
2. **Practice next**: step by step code cells, each one commented so you know exactly what it does.
3. **Recap and exercises**: short summary plus "try it yourself" ideas.

| Part | Topic | What you will build |
|---|---|---|
| 0 | Core concepts and setup | Mental model of ADK 2.0, environment, helper utilities |
| 1 | GenAI prompt based assistant | Persona assistant, multi-turn memory, dynamic prompts, structured output, few-shot classifier |
| 2 | Workflow agent (Graph API) | Sequential pipeline, state passing, conditional routing, parallel fan-out with `JoinNode`, nested workflows |
| 3 | RAG and grounding | Local knowledge-base RAG graph with a relevance gate and citations, Google Search grounding, hybrid grounding |
| 4 | Agentic patterns (ADK standard approach) | Tool use (ReAct), reflection loop, plan-and-execute |
| 5 | Multi-agent solution | LLM coordinator with specialists, parallel expert panel, capstone hybrid support system |

**Prerequisites:** basic Python, Python 3.10 or newer, and a **Google Cloud project** with billing enabled and the Vertex AI API turned on. **No API key is needed**: the notebook authenticates with your Google account through Application Default Credentials (ADC) and calls Gemini through **Gemini Enterprise Agent Platform (formerly Vertex AI)** using your project id.

> **Tip:** Run the cells in order. Later parts reuse helpers, tools, and the knowledge base created in earlier parts.

---
# Part 0: Core Concepts of ADK 2.0

## 0.1 What is ADK?

The **Agent Development Kit (ADK)** is Google's open-source, code-first framework for building, evaluating, and deploying AI agents. You write normal Python; ADK handles model calls, tool execution, conversation history, sessions, and event streaming.

## 0.2 What is new in ADK 2.0?

ADK 2.0 introduces a **graph-based workflow runtime**. Instead of putting your whole business process into one long prompt and hoping the model follows it, you describe the process as a **graph**:

- **Nodes** are units of work. A node can be a plain Python function, an AI `Agent`, a tool, or even another `Workflow`.
- **Edges** connect nodes and decide what runs next (in sequence, conditionally, in parallel, or in a loop).

```
 Prompt-only agent                      Graph workflow
 -----------------                      --------------
 "Step 1 do X, then if Y do Z,          START --> [fn: clean] --> [agent: classify] --> [fn: router]
  otherwise ... and never forget..."                                                   |        |
  (model must follow everything)                                                  [agent A] [agent B]
```

The key benefit: **deterministic steps run as code (fast, free, predictable), and the LLM is used only where reasoning is actually needed.**

## 0.3 Building blocks you will use

| Concept | Import | What it is |
|---|---|---|
| `Agent` | `from google.adk import Agent` | An LLM-powered worker with an `instruction`, optional `tools`, `output_schema`, and `sub_agents` |
| `Workflow` | `from google.adk import Workflow` | A graph of nodes connected by `edges` |
| `START` | `from google.adk.workflow import START` | The entry point of every graph |
| Function node | any Python function | Receives `node_input` (and optionally `ctx`), returns a value or an `Event` |
| `Event` | `from google.adk import Event` | What a node emits. Key fields: `output`, `message`, `route`, `state` |
| `JoinNode` | `from google.adk.workflow import JoinNode` | Waits for parallel branches and merges their outputs into a dict |
| `Runner` | `from google.adk import Runner` | Executes an agent or workflow and streams events |
| `InMemorySessionService` | `from google.adk.sessions import ...` | Stores sessions (conversation history plus state) in memory |
| Session state | `ctx.state` | A small key-value "whiteboard" shared across nodes and turns |

## 0.4 The three data channels of a node

This is the number one source of confusion for new ADK users, so learn it now:

| Channel | Travels to | Use it for |
|---|---|---|
| `output` | The **next node** in the graph | Passing data along the pipeline (like a conveyor belt) |
| `message` | The **user** | Showing text to the person using your app |
| `state` | **Everyone** in the session (via `ctx.state`) | Small values many nodes need: ids, counters, flags |

Plus one control field: `route`, which selects the conditional edge to follow.

## 0.5 Edge syntax cheat sheet

```python
# Sequence: runs A then B then C
edges=[(START, node_a, node_b, node_c)]

# Conditional routing: router returns Event(route="X") or Event(route="Y")
edges=[(START, router), (router, {"X": handler_x, "Y": handler_y})]

# Parallel fan-out and join
join = JoinNode(name="join")
edges=[(START, task_1, join), (START, task_2, join), (join, final_step)]

# Loop: a route pointing back to an earlier node (always add an exit condition!)
edges=[(START, writer, critic), (critic, {"REVISE": writer, "DONE": publish})]
```

## 0.6 Agent modes (important inside graphs)

| Mode | User interaction | Returns control | Where to use |
|---|---|---|---|
| `chat` (default) | Full conversation | Manually, via transfer | Standalone chat assistants |
| `task` | Clarifying questions only | Automatically when done | Sub-agents that collect information |
| `single_turn` | None | Immediately with the result | **Agents inside graph workflows**, parallel specialists |

> In this tutorial every agent placed inside a `Workflow` uses `mode="single_turn"`. The root agent you hand to a `Runner` does not set a mode.

## 0.7 About Gemini 3.5 Flash

- Model id: **`gemini-3.5-flash`**. It is Google's Flash model designed for agentic, multi-step workflows at high speed and lower cost.
- In this notebook it is served from **your Google Cloud project** (Gemini Enterprise Agent Platform, formerly Vertex AI) in the `us-central1` region, so usage is billed to that project.
- Default thinking level is **medium**. You can lower it to `low` for faster, cheaper responses on simple steps.
- Google's migration guidance for 3.5 Flash recommends not tuning `temperature`, `top_p`, or `top_k`; use `thinking_level` instead.

---
## 0.8 Practical Setup

### Step 1: Install the libraries

In [6]:
# ------------------------------------------------------------------
# Install everything this tutorial needs.
#   google-adk>=2.0.0 : Agent Development Kit with the Graph (Workflow) runtime
#   google-genai      : Google Gen AI SDK (Content, Part, config types)
#   scikit-learn      : lightweight TF-IDF retriever used in the RAG section
#   pydantic          : typed schemas for structured input and output
# After installing, restart the kernel if a previous ADK 1.x was loaded.
# ------------------------------------------------------------------
%pip install --quiet --upgrade "google-adk>=2.0.0" google-genai scikit-learn pydantic

Note: you may need to restart the kernel to use updated packages.


In [7]:
# ------------------------------------------------------------------
# Verify installed versions. google-adk must show 2.x.
# If it shows 1.x: Kernel > Restart, then run this cell again.
# ------------------------------------------------------------------
import importlib.metadata as md

for pkg in ["google-adk", "google-genai", "scikit-learn", "pydantic"]:
    try:
        print(f"{pkg:14s} {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg:14s} NOT INSTALLED")

google-adk     2.9.1
google-genai   2.24.0
scikit-learn   1.9.1
pydantic       2.13.5


### Step 2: Google Cloud project setup (no API key)

This notebook calls Gemini through **Vertex AI in your Google Cloud project**. There is no API key. Your identity comes from the Google sign-in you completed on the VM (README Step 2), and the project settings come from the **`.env` file** in the same folder as this notebook:

```
# Use Vertex AI (your Google Cloud project) rather than a personal API key.
GOOGLE_GENAI_USE_VERTEXAI=TRUE

# Your Google Cloud project ID.
GOOGLE_CLOUD_PROJECT=bdc-tred-fde-p01

# The region the model is called in.
GOOGLE_CLOUD_LOCATION=us-central1
```

The next three cells:

1. **Load `.env`** and apply the settings (nothing to edit).
2. **Check your sign-in** (Application Default Credentials).
3. **Make one test call** to Gemini to prove everything works.

> To use a different project or region later, edit `.env`, click **Restart**, and run the setup cells again.

In [8]:
# ------------------------------------------------------------------
# Load project settings from the .env file next to this notebook.
# ------------------------------------------------------------------
import os
from pathlib import Path
from dotenv import load_dotenv   # python-dotenv is installed together with google-adk

ENV_FILE = Path(".env")
if ENV_FILE.exists():
    load_dotenv(ENV_FILE, override=True)
    print(f"Loaded settings from {ENV_FILE.resolve()}")
else:
    print("No .env file found next to the notebook; using the default values below.")

# Settings (defaults match the tutorial .env file).
PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "bdc-tred-fde-shared")
LOCATION = os.environ.get("GOOGLE_CLOUD_LOCATION", "us-central1")
MODEL = "gemini-3.5-flash"

# Use Vertex AI through the Google Cloud project (not an API key).
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION

# Remove anything that could override project authentication.
os.environ.pop("GOOGLE_GENAI_USE_ENTERPRISE", None)
for key_var in ("GOOGLE_API_KEY", "GEMINI_API_KEY"):
    if os.environ.pop(key_var, None):
        print(f"Removed {key_var} from this session (using project credentials instead).")

assert PROJECT_ID, "GOOGLE_CLOUD_PROJECT is empty. Check the .env file."
print(f"Project : {PROJECT_ID}\nLocation: {LOCATION}\nModel   : {MODEL}")

Loaded settings from /home/tren_admin_s1/adk2-tutorial/.env
Project : bdc-tred-fde-shared
Location: global
Model   : gemini-3.5-flash


In [9]:
# ------------------------------------------------------------------
# Verify Application Default Credentials (ADC): the Google sign-in saved
# on the VM by `gcloud auth login ... --update-adc` (README Step 2).
# ------------------------------------------------------------------
import google.auth
from google.auth.exceptions import DefaultCredentialsError

try:
    credentials, adc_project = google.auth.default(
        scopes=["https://www.googleapis.com/auth/cloud-platform"])
    print("ADC credentials found:", type(credentials).__name__)
    print("ADC default project  :", adc_project)
    if adc_project and adc_project != PROJECT_ID:
        print(f"Note: ADC default project differs; this notebook will use PROJECT_ID={PROJECT_ID}.")
except DefaultCredentialsError:
    print("No credentials found. In the VS Code terminal run:\n"
          "  gcloud auth login fdetrainer@bluelabs.studio --update-adc --no-launch-browser\n"
          f"  gcloud auth application-default set-quota-project {PROJECT_ID}\n"
          "Then click Restart and run the setup cells again.")
    raise

ADC credentials found: Credentials
ADC default project  : bdc-tred-fde-shared


In [10]:
# ------------------------------------------------------------------
# Connectivity test: one direct Gemini call through your project (no ADK yet).
# Friendly hints are printed for the most common setup problems.
# ------------------------------------------------------------------
from google import genai

client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

try:
    reply = client.models.generate_content(model=MODEL, contents="Reply with exactly: project auth works")
    print("Gemini says:", reply.text)
except Exception as exc:
    msg = str(exc)
    print("Call failed:", msg[:300], "\n")
    if "SERVICE_DISABLED" in msg or "has not been used" in msg:
        print(f"Hint: enable the API:  gcloud services enable aiplatform.googleapis.com --project {PROJECT_ID}")
    elif "PERMISSION_DENIED" in msg or "403" in msg:
        print("Hint: your account needs the Vertex AI User role (roles/aiplatform.user) on this project,"
              " and billing must be enabled.")
    elif "NOT_FOUND" in msg or "404" in msg:
        print(f"Hint: check PROJECT_ID and LOCATION in .env, and that {MODEL} is enabled in {LOCATION}.")
    elif "quota" in msg.lower() or "429" in msg:
        print(f"Hint: set the quota project:  gcloud auth application-default set-quota-project {PROJECT_ID}")
    raise

Gemini says: project auth works


### Step 3: Imports and global settings

In [11]:
# ------------------------------------------------------------------
# Core imports used across the whole tutorial.
# ------------------------------------------------------------------
import asyncio
import ast
import json
import logging
import operator
import re
import time
import warnings
from typing import Any, Literal

from pydantic import BaseModel, Field
from google.genai import types                        # Content / Part / config types

from google.adk import Agent, Workflow, Runner, Event  # ADK 2.0 top-level API
from google.adk.workflow import START, JoinNode        # Graph primitives
from google.adk.sessions import InMemorySessionService # Session storage for local runs
from google.adk.tools import google_search             # Built-in Google Search grounding tool

# Keep the notebook output readable by silencing library warnings and info logs.
warnings.filterwarnings("ignore")
logging.getLogger("google_genai").setLevel(logging.ERROR)
logging.getLogger("google_adk").setLevel(logging.ERROR)

# MODEL, PROJECT_ID and LOCATION come from the project configuration cell above.
APP_NAME = "adk2_graph_tutorial"

print(f"ADK imports OK. Using {MODEL} in project {os.environ['GOOGLE_CLOUD_PROJECT']} ({os.environ['GOOGLE_CLOUD_LOCATION']})")

ADK imports OK. Using gemini-3.5-flash in project bdc-tred-fde-shared (global)


### Step 4: Helper utilities

Two small helpers keep every example short:

- `to_text()` / `to_dict()` convert whatever a node receives (Content, dict, pydantic model, JSON string) into text or a dict.
- `run()` creates a `Runner` for an `Agent` **or** a `Workflow`, sends a message, prints the event stream (agent replies, tool calls, node outputs, search sources), and returns the final text.

Understanding `run()` is optional, but reading it once shows you the standard ADK execution loop: **Runner -> Session -> run_async -> events**.

In [12]:
# ------------------------------------------------------------------
# Conversion helpers: nodes may receive different data types.
# ------------------------------------------------------------------
def to_text(value: Any) -> str:
    """Turn Content, pydantic models, dicts, lists, or anything else into plain text."""
    if value is None:
        return ""
    if isinstance(value, types.Content):
        # Join only visible text parts (skip model "thought" parts).
        return "".join(p.text for p in (value.parts or [])
                       if getattr(p, "text", None) and not getattr(p, "thought", False))
    if isinstance(value, BaseModel):
        return value.model_dump_json(indent=2)
    if isinstance(value, (dict, list)):
        return json.dumps(value, indent=2, default=str)
    return str(value)


def to_dict(value: Any) -> dict:
    """Best effort conversion of structured agent output into a Python dict."""
    if isinstance(value, dict):
        return value
    if isinstance(value, BaseModel):
        return value.model_dump()
    text = to_text(value).strip()
    text = re.sub(r"^```(?:json)?|```$", "", text, flags=re.MULTILINE).strip()  # strip code fences
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return {"raw": text}


# ------------------------------------------------------------------
# Runner helper: one cached Runner (with its own session store) per agent/workflow.
# ------------------------------------------------------------------
_runners: dict[int, Runner] = {}


def get_runner(target) -> Runner:
    """Create the Runner once. Workflows use node=..., plain agents use agent=..."""
    key = id(target)
    if key not in _runners:
        service = InMemorySessionService()
        if isinstance(target, Workflow):
            _runners[key] = Runner(node=target, app_name=APP_NAME, session_service=service)
        else:
            _runners[key] = Runner(agent=target, app_name=APP_NAME, session_service=service)
    return _runners[key]


async def run(target, message: str, *, session_id: str = "session_1", user_id: str = "learner",
              state: dict | None = None, show_outputs: bool = False, show_tools: bool = True,
              show_sources: bool = False) -> str:
    """Send one user message to an Agent or Workflow and pretty-print the event stream.

    session_id   : reuse the same id to continue a conversation (memory)
    state        : initial session state (or a state update if the session already exists)
    show_outputs : also print node-to-node `output` values (great for learning graphs)
    show_tools   : print tool calls and tool results
    show_sources : print Google Search grounding sources when present
    """
    runner = get_runner(target)
    service = runner.session_service

    # 1) Get or create the session. New sessions start with `state`.
    session = await service.get_session(app_name=APP_NAME, user_id=user_id, session_id=session_id)
    state_delta = None
    if session is None:
        await service.create_session(app_name=APP_NAME, user_id=user_id,
                                     session_id=session_id, state=state or {})
    elif state:
        state_delta = state  # existing session: apply state as an update

    # 2) Wrap the user's text in a Content object (the standard ADK message format).
    new_message = types.Content(role="user", parts=[types.Part(text=message)])
    kwargs = dict(user_id=user_id, session_id=session_id, new_message=new_message)
    if state_delta:
        kwargs["state_delta"] = state_delta

    print(f"USER > {message}\n" + "-" * 80)
    final_text, last_output = "", None

    # 3) Stream events as the agent/workflow runs.
    async for event in runner.run_async(**kwargs):
        author = event.author or "?"
        if event.content and event.content.parts:
            for part in event.content.parts:
                if getattr(part, "thought", False):
                    continue  # hide internal reasoning parts
                if part.function_call and show_tools:
                    print(f"  [tool call]   {author} -> {part.function_call.name}({dict(part.function_call.args or {})})")
                elif part.function_response and show_tools:
                    print(f"  [tool result] {part.function_response.name} -> {part.function_response.response}")
                elif part.text:
                    final_text = part.text
                    print(f"[{author}]: {part.text.strip()}\n")
        if event.output is not None:
            last_output = event.output
            if show_outputs:
                print(f"  [node output from {author}] {to_text(event.output)[:400]}\n")
        if show_sources and event.grounding_metadata and event.grounding_metadata.grounding_chunks:
            print("  [sources]")
            for chunk in event.grounding_metadata.grounding_chunks:
                if chunk.web:
                    print(f"   - {chunk.web.title}: {chunk.web.uri}")

    print("=" * 80)
    return final_text or to_text(last_output)

print("Helpers ready.")

Helpers ready.


### Step 5: Smoke test

If this cell prints a friendly sentence, your project credentials, model access, and ADK install all work through ADK (not just the direct SDK call above).

> Jupyter supports top-level `await`, so we call `await run(...)` directly. In a regular `.py` script you would wrap it with `asyncio.run(...)`.

In [13]:
# ------------------------------------------------------------------
# Smallest possible agent: a name, a model, and an instruction.
# ------------------------------------------------------------------
hello_agent = Agent(
    name="hello_agent",
    model=MODEL,
    instruction="Reply with one short, friendly sentence.",
)

await run(hello_agent, "Say hello to someone learning ADK 2.0!")

USER > Say hello to someone learning ADK 2.0!
--------------------------------------------------------------------------------
[hello_agent]: Welcome to the exciting world of ADK 2.0, and happy learning!



'Welcome to the exciting world of ADK 2.0, and happy learning!'

---
# Part 1: GenAI Prompt Based Assistant

## 1.1 Concepts

A **prompt based assistant** is the simplest kind of agent: an LLM plus a carefully written instruction. No graph and no tools yet. Getting this right matters because every node in later parts is built from the same pieces.

### Anatomy of an ADK `Agent`

| Parameter | Purpose |
|---|---|
| `name` | Unique identifier (letters, digits, underscores) |
| `model` | Which LLM to use, here `gemini-3.5-flash` |
| `description` | One line summary. Other agents read this to decide whether to delegate (Part 5) |
| `instruction` | The system prompt: persona, rules, format |
| `output_schema` | A pydantic model that forces JSON output in a fixed shape |
| `output_key` | Saves the agent's final reply into session state under this key |
| `generate_content_config` | Model settings such as thinking level |

### Prompt engineering patterns shown in this part

1. **Persona and rules**: tell the model who it is, what to do, and what never to do.
2. **Conversation memory**: the session keeps history, so follow-up questions work.
3. **Dynamic instructions**: `{state_key}` placeholders are filled from session state at run time.
4. **Structured output**: `output_schema` turns free text into validated data.
5. **Few-shot prompting**: examples inside the instruction teach format and labels.

### How a request flows

```
User message --> Runner --> Session (history + state) --> Agent builds prompt
                                                           (instruction + history + message)
                                                           --> Gemini 3.5 Flash --> Event stream --> You
```

> **Watch out:** ADK treats `{something}` in an instruction as a state placeholder. Avoid literal curly braces in instructions (for example JSON samples). Use `{key?}` if a key might be missing.

## 1.2 Practical: A persona based assistant

### Step 1: Define the assistant

In [14]:
# ------------------------------------------------------------------
# A travel assistant with a clear persona, rules, and output style.
# A well structured instruction (ROLE / RULES / STYLE) makes behavior consistent.
# ------------------------------------------------------------------
travel_assistant = Agent(
    name="travel_assistant",
    model=MODEL,
    description="A friendly assistant that plans short trips.",
    instruction="""
ROLE
You are Wanderly, a practical and friendly travel planning assistant.

RULES
- If critical information is missing, ask at most ONE clarifying question.
- Keep answers under 150 words unless the user asks for more detail.
- Use short bullet points for itineraries, grouped by day.
- Never invent exact prices, opening hours, or bookings. Say "check locally" instead.
- If a question is not about travel, politely steer back to travel topics.

STYLE
Warm, concise, and specific. End with one helpful tip.
""",
)
print("Agent created:", travel_assistant.name)

Agent created: travel_assistant


### Step 2: Ask a single question

In [15]:
# ------------------------------------------------------------------
# One question in a brand new session.
# ------------------------------------------------------------------
await run(travel_assistant, "Plan a relaxed 2-day trip to Lisbon for a first-time visitor.",
          session_id="trip_chat")

USER > Plan a relaxed 2-day trip to Lisbon for a first-time visitor.
--------------------------------------------------------------------------------
[travel_assistant]: Welcome to Lisbon! Here is a relaxed 2-day itinerary perfect for your first visit.

**Day 1: Historic Heart & Views**
*   **Morning:** Wander the winding, cobblestone streets of historic **Alfama** at a leisurely pace. 
*   **Afternoon:** Take in panoramic views from **Miradouro das Portas do Sol**, then stroll down to the grand **Praça do Comércio**.
*   **Evening:** Enjoy dinner in **Chiado** and try a traditional *pastel de nata* (custard tart).

**Day 2: Belém & Riverside**
*   **Morning:** Head to the waterfront district of **Belém** (check local transport schedules for the easiest route).
*   **Afternoon:** Admire the **Belém Tower** and **Jerónimos Monastery** from the outside (check opening hours and ticket prices locally if you wish to enter).
*   **Evening:** Relax with a peaceful sunset stroll along the Tagu

'Welcome to Lisbon! Here is a relaxed 2-day itinerary perfect for your first visit.\n\n**Day 1: Historic Heart & Views**\n*   **Morning:** Wander the winding, cobblestone streets of historic **Alfama** at a leisurely pace. \n*   **Afternoon:** Take in panoramic views from **Miradouro das Portas do Sol**, then stroll down to the grand **Praça do Comércio**.\n*   **Evening:** Enjoy dinner in **Chiado** and try a traditional *pastel de nata* (custard tart).\n\n**Day 2: Belém & Riverside**\n*   **Morning:** Head to the waterfront district of **Belém** (check local transport schedules for the easiest route).\n*   **Afternoon:** Admire the **Belém Tower** and **Jerónimos Monastery** from the outside (check opening hours and ticket prices locally if you wish to enter).\n*   **Evening:** Relax with a peaceful sunset stroll along the Tagus River.\n\n**Wanderly Tip:** Lisbon is famously hilly! Wear comfortable walking shoes with good grip, as the beautiful limestone cobblestones can be very slip

### Step 3: Multi-turn conversation (memory)

We reuse the **same `session_id`**. The session stores the earlier exchange, so the assistant understands "that trip".

In [16]:
# ------------------------------------------------------------------
# Follow-up question in the SAME session: the agent remembers the context.
# ------------------------------------------------------------------
await run(travel_assistant, "What should I pack for that trip if it's in November?",
          session_id="trip_chat")

# A different session id starts a fresh conversation with no memory of Lisbon.
await run(travel_assistant, "What should I pack for that trip?", session_id="new_chat")

USER > What should I pack for that trip if it's in November?
--------------------------------------------------------------------------------
[travel_assistant]: November in Lisbon brings mild but changeable autumn weather, so layering is key. Here is what you should pack:

*   **Footwear:** Comfortable walking shoes with good rubber soles. Lisbon's wet cobblestones can be very slippery!
*   **Clothing:** Light layers, including t-shirts, long-sleeve shirts, sweaters, and long pants. 
*   **Outerwear:** A windproof, waterproof jacket or a sturdy travel umbrella, as November is one of Lisbon's rainier months.
*   **Accessories:** Sunglasses (the Atlantic light is bright even in winter) and a small daypack for your layers.

**Wanderly Tip:** Check the local weather forecast just before you depart, as autumn temperatures can fluctuate quickly between sunny warmth and chilly rain.

USER > What should I pack for that trip?
--------------------------------------------------------------------

'I would love to help you pack, but I need a little more information first! \n\n**Where are you traveling to, and what will the weather or main activities be like?** \n\nOnce you let me know, I can give you a tailored list of essentials. \n\n*Tip: No matter the destination, always roll your clothes instead of folding them—it saves an incredible amount of suitcase space and prevents wrinkles!*'

## 1.3 Practical: Dynamic instructions with session state

Placeholders like `{user_name}` are replaced with values from session state before the prompt reaches the model. One agent definition can therefore serve many users differently.

In [17]:
# ------------------------------------------------------------------
# A tutor whose instruction adapts to each learner via state placeholders.
# ------------------------------------------------------------------
coding_tutor = Agent(
    name="coding_tutor",
    model=MODEL,
    instruction="""
You are a patient programming tutor.
The learner's name is {user_name}. Their skill level is {skill_level}.

- Address the learner by name.
- Match vocabulary and depth to the {skill_level} level.
- Answer in at most 120 words and include a tiny code example.
- Finish with exactly one practice question.
""",
)

# Same question, two learners, two different initial states.
await run(coding_tutor, "What is a Python decorator?", session_id="tutor_beginner",
          state={"user_name": "Maya", "skill_level": "beginner"})

await run(coding_tutor, "What is a Python decorator?", session_id="tutor_advanced",
          state={"user_name": "Leo", "skill_level": "advanced"})

USER > What is a Python decorator?
--------------------------------------------------------------------------------
[coding_tutor]: Hi Maya! Think of a Python decorator as a gift wrapper. It lets you "wrap" another function to add extra behavior to it, without changing its original code. In Python, we apply decorators using the `@` symbol.

Here is a tiny example:

```python
def my_decorator(func):
    def wrapper():
        print("🎁 Wrapping...")
        func()
    return wrapper

@my_decorator
def greet():
    print("Hello Maya!")

greet()
```

When you call `greet()`, it automatically runs the wrapper code first!

For your practice: **If you wanted a decorator to run a function twice, how many times would you need to call `func()` inside the `wrapper()`?**

USER > What is a Python decorator?
--------------------------------------------------------------------------------
[coding_tutor]: Hello Leo. 

In Python, a decorator is a higher-order function that takes another function (or cl

'Hello Leo. \n\nIn Python, a decorator is a higher-order function that takes another function (or class) as an argument, extends its behavior without explicit modification, and returns a new callable. Leveraged for elegant metaprogramming, decorators exploit Python’s first-class functions and lexical closures.\n\nHere is a lightweight example:\n\n```python\ndef uppercase(func):\n    def wrapper(*args, **kwargs):\n        return func(*args, **kwargs).upper()\n    return wrapper\n\n@uppercase\ndef greet(name): \n    return f"hello, {name}"\n```\n\nThis pattern is crucial for implementing cross-cutting concerns like logging, caching, and access control.\n\n**Your practice question:**\nHow would you design a stateful decorator that accepts configuration arguments (e.g., limiting a function to `N` executions) while preserving the original function\'s metadata?'

## 1.4 Practical: Structured output with `output_schema`

Free text is great for people but awkward for programs. With `output_schema`, Gemini must return JSON that matches a pydantic model, which you can then validate and use in code.

In [18]:
# ------------------------------------------------------------------
# Step 1: Describe the data you want as pydantic models.
# Field descriptions act as extra instructions for the model.
# ------------------------------------------------------------------
class ActionItem(BaseModel):
    owner: str = Field(description="Person responsible")
    task: str = Field(description="What needs to be done")
    due: str | None = Field(default=None, description="Deadline if mentioned, else null")


class MeetingNotes(BaseModel):
    title: str = Field(description="Short title for the meeting")
    decisions: list[str] = Field(description="Decisions that were made")
    action_items: list[ActionItem]
    sentiment: Literal["positive", "neutral", "negative"]


# ------------------------------------------------------------------
# Step 2: Create an agent that must answer in that schema.
# output_key also stores the JSON result in session state as "meeting_notes".
# ------------------------------------------------------------------
notes_extractor = Agent(
    name="notes_extractor",
    model=MODEL,
    instruction="Extract structured meeting notes from the transcript the user provides.",
    output_schema=MeetingNotes,
    output_key="meeting_notes",
)

In [19]:
# ------------------------------------------------------------------
# Step 3: Run it and validate the JSON into a typed Python object.
# ------------------------------------------------------------------
transcript = """
Priya: Thanks all. We agreed to launch the mobile app beta on March 3rd.
Tom: I'll finish the onboarding screens by Friday.
Priya: Great. Sam, can you prepare the beta tester email list by next Tuesday?
Sam: Sure. Also, we decided to postpone the dark mode feature to version 2.
Priya: Good progress everyone, this is looking really solid.
"""

raw = await run(notes_extractor, transcript, session_id="notes_1")
notes = MeetingNotes.model_validate(to_dict(raw))   # raises if the shape is wrong

print("Title     :", notes.title)
print("Sentiment :", notes.sentiment)
for item in notes.action_items:
    print(f"  - {item.owner}: {item.task} (due: {item.due})")

USER > 
Priya: Thanks all. We agreed to launch the mobile app beta on March 3rd.
Tom: I'll finish the onboarding screens by Friday.
Priya: Great. Sam, can you prepare the beta tester email list by next Tuesday?
Sam: Sure. Also, we decided to postpone the dark mode feature to version 2.
Priya: Good progress everyone, this is looking really solid.

--------------------------------------------------------------------------------
[notes_extractor]: {"title":"Mobile App Beta Launch Planning","decisions":["Launch the mobile app beta on March 3rd","Postpone the dark mode feature to version 2"],"action_items":[{"owner":"Tom","task":"Finish the onboarding screens","due":"Friday"},{"owner":"Sam","task":"Prepare the beta tester email list","due":"next Tuesday"}],"sentiment":"positive"}

Title     : Mobile App Beta Launch Planning
Sentiment : positive
  - Tom: Finish the onboarding screens (due: Friday)
  - Sam: Prepare the beta tester email list (due: next Tuesday)


## 1.5 Practical: Few-shot prompting and model settings

Examples inside the instruction ("shots") teach the model the exact labels and format. We also set `thinking_level="low"` because classification is simple, which makes it faster and cheaper.

In [20]:
# ------------------------------------------------------------------
# Few-shot sentiment and intent classifier.
# thinking_level="low" : simple task, so spend less reasoning effort.
# ------------------------------------------------------------------
review_classifier = Agent(
    name="review_classifier",
    model=MODEL,
    generate_content_config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(thinking_level="low"),
    ),
    instruction="""
Classify a product review. Reply on ONE line in this exact format:
SENTIMENT | TOPIC | URGENT

Allowed SENTIMENT: POSITIVE, NEGATIVE, MIXED
Allowed TOPIC: DELIVERY, QUALITY, PRICE, SUPPORT, OTHER
URGENT is YES only if the customer reports a safety issue or demands a refund.

Examples:
Review: "Arrived two days early and works perfectly!"
Answer: POSITIVE | DELIVERY | NO

Review: "The charger got extremely hot and melted. I want my money back."
Answer: NEGATIVE | QUALITY | YES

Review: "Great sound, but honestly too expensive for what it is."
Answer: MIXED | PRICE | NO
""",
)

reviews = [
    "Support replied in five minutes and fixed my issue. Impressed!",
    "Package was crushed and the glass is cracked. Refund please.",
    "Nice design, though the battery could last longer.",
]

# Each review gets its own session so earlier answers do not influence later ones.
for i, review in enumerate(reviews):
    await run(review_classifier, review, session_id=f"review_{i}")

USER > Support replied in five minutes and fixed my issue. Impressed!
--------------------------------------------------------------------------------
[review_classifier]: POSITIVE | SUPPORT | NO

USER > Package was crushed and the glass is cracked. Refund please.
--------------------------------------------------------------------------------
[review_classifier]: NEGATIVE | DELIVERY | YES

USER > Nice design, though the battery could last longer.
--------------------------------------------------------------------------------
[review_classifier]: MIXED | QUALITY | NO



## Part 1 Recap

- An `Agent` is `name + model + instruction`, optionally with `output_schema`, `output_key`, and model config.
- **Sessions** give memory: same `session_id` means same conversation.
- **State placeholders** (`{key}`) personalize one agent for many users.
- **`output_schema`** turns responses into validated data your code can trust.
- **Few-shot examples** plus a low thinking level give fast, consistent classification.

**Try it yourself**
1. Add a rule to `travel_assistant` that it must always ask for the traveler's budget first.
2. Add a `priority: Literal["low","medium","high"]` field to `ActionItem` and re-run the extractor.
3. Add a fourth label `NEUTRAL` to the classifier and write an example for it.

---
# Part 2: Workflow Agent with the Graph API

## 2.1 Concepts

### Why graphs?

As prompts grow ("first do this, then if that, otherwise..."), models skip steps or improvise. A **Workflow** moves the *process* into code while keeping the *reasoning* in agents:

| Prompt-only agent | Graph workflow |
|---|---|
| Model decides the order of steps | You define the order with edges |
| Every step costs tokens | Function nodes cost zero tokens |
| Hard to debug which step failed | Each node emits its own events |
| Loops and branches are implicit | Loops and branches are explicit and testable |

### Node types

| Node | How to create | Typical job |
|---|---|---|
| Function node | Any Python function `def f(node_input, ctx)` | Parsing, validation, API calls, routing |
| Agent node | `Agent(..., mode="single_turn")` | Summarize, classify, write, reason |
| Join node | `JoinNode(name=...)` | Merge parallel branches |
| Nested workflow | A `Workflow` used inside another | Reusable sub-process |

### Function node signature

```python
def my_node(node_input, ctx):     # both parameters are optional
    # node_input : output of the previous node (the user's Content for the first node)
    # ctx.state  : session state dictionary (read and write)
    return "value"                # plain return value becomes `output` for the next node
    # or: return Event(output=..., route=..., message=...)
```

### Patterns in this part

```
2.2 Sequential   START -> clean_input -> summarizer(agent) -> add_footer
2.3 State        START -> capture -> translator(agent) -> save -> back_translator(agent) -> report
2.4 Routing      START -> capture -> classifier(agent) -> router --BILLING--> billing_agent
                                                                --TECHNICAL--> technical_agent
                                                                --SALES--> sales_agent
                                                                --OTHER--> other_handler
2.5 Parallel     START -> fetch_weather  --\
                 START -> fetch_calendar ---> JoinNode -> build_prompt -> briefing_agent
                 START -> fetch_news     --/
2.6 Nested       START -> capture -> [polish_subflow: grammar -> tone] -> finalize
```

## 2.2 Practical: Sequential pipeline

Two function nodes (free) wrap one agent node (the only LLM call).

In [21]:
# ------------------------------------------------------------------
# Node 1 (function, 0 tokens): clean the raw user text and build a precise prompt.
# The first node receives the user's message as a types.Content object.
# ------------------------------------------------------------------
def clean_input(node_input):
    text = " ".join(to_text(node_input).split())          # collapse extra whitespace
    return f"Summarize the following text in exactly 3 bullet points:\n\n{text}"


# ------------------------------------------------------------------
# Node 2 (agent): the only step that calls Gemini.
# Agents inside graphs use mode="single_turn" (no back-and-forth with the user).
# ------------------------------------------------------------------
summarizer = Agent(
    name="summarizer",
    model=MODEL,
    mode="single_turn",
    instruction="You write crisp, factual bullet summaries. Output only the bullets.",
)


# ------------------------------------------------------------------
# Node 3 (function, 0 tokens): post-process and show a message to the user.
# ------------------------------------------------------------------
def add_footer(node_input):
    summary = to_text(node_input).strip()
    return Event(message=f"SUMMARY REPORT\n{summary}\n\n(length: {len(summary.split())} words)")


# ------------------------------------------------------------------
# Wire the graph: one tuple = one chain of nodes executed in order.
# ------------------------------------------------------------------
summary_pipeline = Workflow(
    name="summary_pipeline",
    edges=[(START, clean_input, summarizer, add_footer)],
)

article = """Solar power capacity grew rapidly over the last decade as panel prices fell
sharply.   Many countries now add more solar than any other electricity source each year.
Grid operators, however, must manage midday oversupply and evening demand peaks, which is
driving large investments in battery storage and smarter demand response programs."""

# show_outputs=True prints what each node passes to the next one.
await run(summary_pipeline, article, session_id="seq_1", show_outputs=True)

USER > Solar power capacity grew rapidly over the last decade as panel prices fell
sharply.   Many countries now add more solar than any other electricity source each year.
Grid operators, however, must manage midday oversupply and evening demand peaks, which is
driving large investments in battery storage and smarter demand response programs.
--------------------------------------------------------------------------------
  [node output from summary_pipeline] Summarize the following text in exactly 3 bullet points:

Solar power capacity grew rapidly over the last decade as panel prices fell sharply. Many countries now add more solar than any other electricity source each year. Grid operators, however, must manage midday oversupply and evening demand peaks, which is driving large investments in battery storage and smarter demand response programs.

[summarizer]: * Rapidly falling panel prices have fueled massive growth in solar power capacity over the last decade.
* Solar has become th

'SUMMARY REPORT\n* Rapidly falling panel prices have fueled massive growth in solar power capacity over the last decade.\n* Solar has become the leading source of new annual electricity capacity in numerous nations.\n* To manage daily supply-demand mismatches, grid operators are investing heavily in battery storage and smart demand response.\n\n(length: 50 words)'

## 2.3 Practical: Sharing data with session state

`output` only reaches the **next** node. When a later node needs something from much earlier (like the original text), store it in `ctx.state`. Agent instructions can also read state with `{key}`.

This pipeline does a **round-trip translation quality check**.

In [22]:
# ------------------------------------------------------------------
# Node 1: remember the original text in state, then pass it on.
# ------------------------------------------------------------------
def capture_original(node_input, ctx):
    text = to_text(node_input).strip()
    ctx.state["original_text"] = text        # available to every later node
    return text


# Node 2 (agent): {target_language} is filled from session state.
translator = Agent(
    name="translator",
    model=MODEL,
    mode="single_turn",
    instruction="Translate the user's text into {target_language}. Output only the translation.",
)


# Node 3: keep the translation in state as well.
def save_translation(node_input, ctx):
    ctx.state["translation"] = to_text(node_input).strip()
    return ctx.state["translation"]


# Node 4 (agent): translate back to English so we can compare meaning.
back_translator = Agent(
    name="back_translator",
    model=MODEL,
    mode="single_turn",
    instruction="Translate the user's text into English. Output only the translation.",
)


# Node 5: read everything from state and build a user-facing report.
def translation_report(node_input, ctx):
    return Event(message=(
        f"ORIGINAL    : {ctx.state['original_text']}\n"
        f"TRANSLATION : {ctx.state['translation']}\n"
        f"BACK TO EN  : {to_text(node_input).strip()}"
    ))


translation_check = Workflow(
    name="translation_check",
    edges=[(START, capture_original, translator, save_translation, back_translator, translation_report)],
)

# Initial state sets the target language for this session.
await run(translation_check, "Our new update makes the app twice as fast and uses less battery.",
          session_id="translate_1", state={"target_language": "Spanish"})

USER > Our new update makes the app twice as fast and uses less battery.
--------------------------------------------------------------------------------
[translator]: Nuestra nueva actualización hace que la aplicación sea el doble de rápida y consuma menos batería.

[back_translator]: Our new update makes the app twice as fast and consume less battery.

[translation_check]: ORIGINAL    : Our new update makes the app twice as fast and uses less battery.
TRANSLATION : Nuestra nueva actualización hace que la aplicación sea el doble de rápida y consuma menos batería.
BACK TO EN  : Our new update makes the app twice as fast and consume less battery.



'ORIGINAL    : Our new update makes the app twice as fast and uses less battery.\nTRANSLATION : Nuestra nueva actualización hace que la aplicación sea el doble de rápida y consuma menos batería.\nBACK TO EN  : Our new update makes the app twice as fast and consume less battery.'

## 2.4 Practical: Conditional routing

A router node returns `Event(route="LABEL")`. The edge dictionary maps each label to the node that should run. We combine an **LLM classifier** (understands language) with a **deterministic router** (validates the label and guarantees a safe fallback).

In [23]:
# ------------------------------------------------------------------
# Node 1: save the customer's message so the chosen specialist gets the full text.
# ------------------------------------------------------------------
def capture_message(node_input, ctx):
    ctx.state["customer_message"] = to_text(node_input).strip()
    return ctx.state["customer_message"]


# Node 2 (agent): classify into one label. Low thinking level: it is a simple task.
ticket_classifier = Agent(
    name="ticket_classifier",
    model=MODEL,
    mode="single_turn",
    generate_content_config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(thinking_level="low")),
    instruction=("Classify the customer message into exactly ONE label: "
                 "BILLING, TECHNICAL, SALES, or OTHER. Reply with the label only."),
)

VALID_ROUTES = ["BILLING", "TECHNICAL", "SALES", "OTHER"]


# Node 3 (function): never trust raw LLM text for control flow. Validate it.
def route_ticket(node_input, ctx):
    raw_label = to_text(node_input).strip().upper()
    label = next((r for r in VALID_ROUTES if r in raw_label), "OTHER")   # safe fallback
    print(f"  [router] classifier said {raw_label!r} -> route {label}")
    # output = the original message for the specialist, route = which edge to follow
    return Event(output=ctx.state["customer_message"], route=label)


# Specialist agents (only ONE of them runs per request).
billing_agent = Agent(name="billing_agent", model=MODEL, mode="single_turn",
    instruction="You are a billing specialist. Help with invoices, refunds, and charges in under 60 words.")
technical_agent = Agent(name="technical_agent", model=MODEL, mode="single_turn",
    instruction="You are a technical support engineer. Give up to 3 concrete troubleshooting steps.")
sales_agent = Agent(name="sales_agent", model=MODEL, mode="single_turn",
    instruction="You are a helpful sales advisor. Explain plan options briefly and invite a demo.")


# A function node can also be a branch target (no LLM cost for simple replies).
def other_handler(node_input):
    return Event(message="Thanks for reaching out! A team member will reply within 24 hours.")


support_router = Workflow(
    name="support_router",
    edges=[
        (START, capture_message, ticket_classifier, route_ticket),
        (route_ticket, {
            "BILLING": billing_agent,
            "TECHNICAL": technical_agent,
            "SALES": sales_agent,
            "OTHER": other_handler,
        }),
    ],
)

In [24]:
# ------------------------------------------------------------------
# Test several messages. Each one follows a different branch of the same graph.
# ------------------------------------------------------------------
test_messages = [
    "I was charged twice for my subscription this month.",
    "The desktop app freezes every time I open a large file.",
    "Do you offer discounts for teams of 50 people?",
    "I just wanted to say your team was lovely at the conference!",
]

for i, msg in enumerate(test_messages):
    await run(support_router, msg, session_id=f"route_{i}")

USER > I was charged twice for my subscription this month.
--------------------------------------------------------------------------------
[ticket_classifier]: BILLING

  [router] classifier said 'BILLING' -> route BILLING
[billing_agent]: I apologize for the double charge! This sometimes happens due to a processing error or a temporary authorization hold. 

Please share your account email or the invoice numbers. I will investigate immediately and issue a refund for the duplicate charge right away. You should see the funds back in 3-5 business days.

USER > The desktop app freezes every time I open a large file.
--------------------------------------------------------------------------------
[ticket_classifier]: TECHNICAL

  [router] classifier said 'TECHNICAL' -> route TECHNICAL
[technical_agent]: Here are 3 concrete steps to troubleshoot and resolve the freezing issue:

1. **Move the file to your local drive:** 
   If the file is stored on a network drive, external hard drive, or sy

## 2.5 Practical: Parallel fan-out with `JoinNode`

Independent steps should not wait for each other. Several edges from `START` run **concurrently**; a `JoinNode` waits for all of them and outputs a dict keyed by node name.

> **Rule:** every branch feeding a `JoinNode` must produce an output, otherwise the join cannot complete.

In [25]:
# ------------------------------------------------------------------
# Three async "data fetchers". Each simulates a 1 second API call.
# ------------------------------------------------------------------
async def fetch_weather(node_input):
    await asyncio.sleep(1)
    return {"city": "Lisbon", "forecast": "Sunny, 24 C", "rain_chance": "10%"}


async def fetch_calendar(node_input):
    await asyncio.sleep(1)
    return {"meetings": ["09:30 Stand-up", "13:00 Client demo", "16:00 1:1 with manager"]}


async def fetch_news(node_input):
    await asyncio.sleep(1)
    return {"headlines": ["City opens new metro line", "Tech fair starts this weekend"]}


# The join node merges all three outputs into one dict.
briefing_join = JoinNode(name="briefing_join")


# Turn the merged dict into a prompt for the agent.
def build_briefing_prompt(node_input):
    data = node_input   # {"fetch_weather": {...}, "fetch_calendar": {...}, "fetch_news": {...}}
    return ("Write a cheerful 4 sentence morning briefing using this data:\n"
            + json.dumps(data, indent=2))


briefing_agent = Agent(
    name="briefing_agent", model=MODEL, mode="single_turn",
    instruction="You are a personal assistant who writes short, upbeat morning briefings.",
)

morning_briefing = Workflow(
    name="morning_briefing",
    edges=[
        (START, fetch_weather, briefing_join),      # branch 1
        (START, fetch_calendar, briefing_join),     # branch 2
        (START, fetch_news, briefing_join),         # branch 3
        (briefing_join, build_briefing_prompt, briefing_agent),
    ],
)

start = time.time()
await run(morning_briefing, "Good morning!", session_id="brief_1")
print(f"Total time: {time.time() - start:.1f}s (the three 1s fetches ran in parallel)")

USER > Good morning!
--------------------------------------------------------------------------------
[briefing_agent]: Good morning, and get ready for a gorgeous, sunny 24°C day in Lisbon with a mere 10% chance of rain! Your fabulous schedule today features a 9:30 stand-up, an exciting client demo at 13:00, and a friendly 1:1 with your manager at 16:00. In local news, you can look forward to exploring the city's brand-new metro line and checking out the big tech fair starting this weekend. You are going to do amazing things today, so let’s make it a great one!

Total time: 8.7s (the three 1s fetches ran in parallel)


## 2.6 Practical: Nested workflows

A `Workflow` is itself a node, so you can package a reusable sub-process and drop it into a bigger graph. We use a **factory function** so each parent graph gets its own fresh copy.

In [26]:
# ------------------------------------------------------------------
# Factory for a reusable 2-step "polish" sub-workflow.
# Factories avoid reusing the same node objects in multiple parent graphs.
# ------------------------------------------------------------------
def make_polish_subflow():
    grammar_fixer = Agent(name="grammar_fixer", model=MODEL, mode="single_turn",
        instruction="Fix grammar and spelling. Keep the meaning. Output only the corrected text.")
    tone_adjuster = Agent(name="tone_adjuster", model=MODEL, mode="single_turn",
        instruction="Rewrite the text in a friendly, professional tone. Output only the text.")
    return Workflow(name="polish_subflow", edges=[(START, grammar_fixer, tone_adjuster)])


def capture_draft(node_input):
    return to_text(node_input).strip()


def finalize_email(node_input):
    return Event(message=f"READY TO SEND:\n\n{to_text(node_input).strip()}")


# The parent graph treats the whole sub-workflow as ONE node.
email_polisher = Workflow(
    name="email_polisher",
    edges=[(START, capture_draft, make_polish_subflow(), finalize_email)],
)

await run(email_polisher,
          "hey team, the report are late becuz data was wrong. send fix asap or we miss deadline",
          session_id="polish_1")

USER > hey team, the report are late becuz data was wrong. send fix asap or we miss deadline
--------------------------------------------------------------------------------
[grammar_fixer]: Hey team, the report is late because the data was wrong. Send the fix ASAP or we will miss the deadline.

[tone_adjuster]: Hi team, 

We are currently experiencing a slight delay with the report due to some inaccuracies in the data. Could you please send over the corrected information as soon as possible? This will help us ensure we stay on track and meet our upcoming deadline. 

Thanks so much for your help with this!

[email_polisher]: READY TO SEND:

Hi team, 

We are currently experiencing a slight delay with the report due to some inaccuracies in the data. Could you please send over the corrected information as soon as possible? This will help us ensure we stay on track and meet our upcoming deadline. 

Thanks so much for your help with this!



'READY TO SEND:\n\nHi team, \n\nWe are currently experiencing a slight delay with the report due to some inaccuracies in the data. Could you please send over the corrected information as soon as possible? This will help us ensure we stay on track and meet our upcoming deadline. \n\nThanks so much for your help with this!'

## Part 2 Recap

- A `Workflow` is a graph of **nodes** joined by **edges**, starting at `START`.
- **Function nodes** are free and deterministic. **Agent nodes** (`mode="single_turn"`) add reasoning.
- `output` goes to the next node, `message` goes to the user, `ctx.state` is shared by everyone.
- `Event(route=...)` plus a dict edge gives **conditional routing**. Always validate LLM labels.
- Multiple edges from one source run **in parallel**; `JoinNode` merges them.
- Workflows can be **nested** as reusable sub-processes.

**Try it yourself**
1. Add a `REFUND` route to `support_router` that goes to a function node asking for the order id.
2. Add a fourth parallel fetcher (for example stock prices) to `morning_briefing`.
3. Insert a word-limit check node after `summarizer` that routes back to it if the summary is too long (a loop!).

---
# Part 3: RAG and Grounding

## 3.1 Concepts

### The problem: hallucination and stale knowledge

LLMs know only what was in their training data. They do not know your company's policies, today's news, or private documents, and they may produce confident but wrong answers. **Grounding** means giving the model trusted information at request time and instructing it to answer from that information.

### Two kinds of grounding

| Type | Source of truth | ADK approach |
|---|---|---|
| **RAG over your data** | Your documents, wikis, databases | Retrieve relevant chunks, then pass them to an agent |
| **Google Search grounding** | The public web, fresh information | Built-in `google_search` tool on an agent |

### The RAG pipeline

```
 INDEXING (once)                           QUERY TIME (every question)
 ---------------                           ---------------------------
 documents --> chunk --> index              question --> retrieve top-k chunks --> relevance gate
                                                                     |                     |
                                                            relevant: augment prompt   none: fallback
                                                                     |
                                                     grounded agent answers + cites sources
```

1. **Chunk**: split documents into small overlapping pieces.
2. **Index**: convert chunks into vectors (here TF-IDF; in production usually embeddings).
3. **Retrieve**: find the chunks most similar to the question.
4. **Gate**: if nothing is relevant enough, do not let the model guess.
5. **Augment and generate**: put the chunks in the prompt; the agent answers and cites chunk ids.

### Why build RAG as a graph?

Retrieval, scoring, and the relevance gate are **deterministic code** (zero tokens, easy to test). Only the final answer uses the LLM. The gate becomes an explicit route instead of a hopeful sentence in a prompt.

```
START -> retrieve_context --ANSWER-->     grounded_answerer (agent)
                          --NO_CONTEXT--> fallback (polite refusal OR web search agent)
```

## 3.2 Practical: Build a small knowledge base

We use a fictional SaaS company, **NimbusCloud**, so the model cannot already know the answers. That proves the answers come from retrieval.

In [27]:
# ------------------------------------------------------------------
# Step 1: Our "documents". In real projects these come from PDFs, wikis, or databases.
# ------------------------------------------------------------------
KNOWLEDGE_BASE = [
    {"id": "refund-policy", "title": "Refund Policy",
     "text": "NimbusCloud offers a full refund within 14 days of purchase for annual plans. "
             "Monthly plans can be cancelled anytime but are not refunded for the current month. "
             "Refunds are processed to the original payment method within 5 to 7 business days. "
             "Enterprise contracts follow the refund terms written in the signed agreement."},
    {"id": "pricing-plans", "title": "Pricing Plans",
     "text": "The Starter plan costs 12 USD per user per month and includes 100 GB of storage. "
             "The Business plan costs 29 USD per user per month and includes 1 TB of storage and SSO. "
             "The Enterprise plan has custom pricing, unlimited storage, and a dedicated success manager. "
             "Annual billing gives a 20 percent discount on Starter and Business plans."},
    {"id": "sla-uptime", "title": "Service Level Agreement",
     "text": "NimbusCloud guarantees 99.9 percent monthly uptime for Business plans and 99.99 percent "
             "for Enterprise plans. If uptime falls below the guarantee, customers receive service credits "
             "of 10 percent of the monthly fee for each 0.1 percent below target, up to 50 percent."},
    {"id": "data-retention", "title": "Data Retention and Deletion",
     "text": "Deleted files stay in the trash for 30 days before permanent deletion. "
             "After an account is closed, all customer data is permanently erased within 60 days. "
             "Enterprise customers can request a certified deletion report from the compliance team."},
    {"id": "security", "title": "Security Features",
     "text": "All data is encrypted with AES-256 at rest and TLS 1.3 in transit. "
             "Two-factor authentication is available on all plans and can be enforced by admins on "
             "Business and Enterprise plans. NimbusCloud is SOC 2 Type II and ISO 27001 certified."},
    {"id": "support-hours", "title": "Customer Support",
     "text": "Starter customers get email support with a 48 hour response target. "
             "Business customers get chat and email support 24 hours a day on weekdays. "
             "Enterprise customers get 24/7 phone support with a 1 hour response time for critical issues."},
]
print(f"Loaded {len(KNOWLEDGE_BASE)} documents.")

Loaded 6 documents.


In [28]:
# ------------------------------------------------------------------
# Step 2: Chunking. Overlap keeps sentences that cross a boundary retrievable.
# ------------------------------------------------------------------
def chunk_text(text: str, max_words: int = 40, overlap: int = 10) -> list[str]:
    words = text.split()
    chunks, start = [], 0
    while start < len(words):
        chunks.append(" ".join(words[start:start + max_words]))
        if start + max_words >= len(words):
            break
        start += max_words - overlap
    return chunks


# ------------------------------------------------------------------
# Step 3: A tiny TF-IDF retriever (no API calls, runs instantly).
# Production upgrade: swap in embeddings + a vector database, keep the same interface.
# ------------------------------------------------------------------
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


class SimpleRetriever:
    def __init__(self, documents: list[dict]):
        self.chunks = []
        for doc in documents:
            for i, piece in enumerate(chunk_text(doc["text"])):
                self.chunks.append({"chunk_id": f"{doc['id']}#{i}", "title": doc["title"], "text": piece})
        # Title is included in the indexed text so "refund" matches the Refund Policy strongly.
        self.vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), sublinear_tf=True)
        self.matrix = self.vectorizer.fit_transform([f"{c['title']} {c['text']}" for c in self.chunks])

    def search(self, query: str, k: int = 3) -> list[dict]:
        scores = cosine_similarity(self.vectorizer.transform([query]), self.matrix)[0]
        best = scores.argsort()[::-1][:k]
        return [{**self.chunks[i], "score": round(float(scores[i]), 3)} for i in best]


retriever = SimpleRetriever(KNOWLEDGE_BASE)
print(f"Indexed {len(retriever.chunks)} chunks.")

Indexed 10 chunks.


In [29]:
# ------------------------------------------------------------------
# Step 4: Test retrieval by itself BEFORE adding an LLM.
# Good RAG debugging habit: check that the right chunks come back.
# ------------------------------------------------------------------
for q in ["How long do refunds take?", "Is two-factor authentication available?", "Who won the football match?"]:
    print(f"\nQ: {q}")
    for hit in retriever.search(q):
        print(f"   {hit['score']:.3f}  {hit['chunk_id']:18s} {hit['text'][:70]}...")


Q: How long do refunds take?
   0.160  refund-policy#0    NimbusCloud offers a full refund within 14 days of purchase for annual...
   0.000  support-hours#0    Starter customers get email support with a 48 hour response target. Bu...
   0.000  security#1         plans. NimbusCloud is SOC 2 Type II and ISO 27001 certified....

Q: Is two-factor authentication available?
   0.357  security#0         All data is encrypted with AES-256 at rest and TLS 1.3 in transit. Two...
   0.000  support-hours#0    Starter customers get email support with a 48 hour response target. Bu...
   0.000  security#1         plans. NimbusCloud is SOC 2 Type II and ISO 27001 certified....

Q: Who won the football match?
   0.000  support-hours#0    Starter customers get email support with a 48 hour response target. Bu...
   0.000  security#1         plans. NimbusCloud is SOC 2 Type II and ISO 27001 certified....
   0.000  security#0         All data is encrypted with AES-256 at rest and TLS 1.3 in transit. Two.

## 3.3 Practical: A RAG workflow with a relevance gate and citations

Notice the unrelated football question scored near zero. We turn that observation into a **route**: below the threshold, the answering agent never runs, so it cannot hallucinate.

In [30]:
RELEVANCE_THRESHOLD = 0.10   # tune this by looking at scores from the test cell above


def make_rag_workflow(name: str = "kb_rag", fallback: str = "refuse") -> Workflow:
    """Build a fresh RAG graph.

    fallback="refuse" : politely say the answer is not in the knowledge base
    fallback="web"    : fall back to a Google Search grounded agent (used in 3.5)
    """

    # Node 1 (function): retrieve, score, and decide the route. Zero tokens.
    def retrieve_context(node_input, ctx):
        question = to_text(node_input).strip()
        hits = [h for h in retriever.search(question, k=3) if h["score"] >= RELEVANCE_THRESHOLD]
        ctx.state["rag_sources"] = [h["chunk_id"] for h in hits]     # keep for auditing
        print(f"  [retriever] {len(hits)} relevant chunk(s): {ctx.state['rag_sources']}")

        if not hits:
            return Event(output=question, route="NO_CONTEXT")

        context = "\n\n".join(f"[{h['chunk_id']}] {h['title']}: {h['text']}" for h in hits)
        prompt = f"CONTEXT:\n{context}\n\nQUESTION: {question}"
        return Event(output=prompt, route="ANSWER")

    # Node 2a (agent): answers ONLY from the provided context, with citations.
    grounded_answerer = Agent(
        name="grounded_answerer",
        model=MODEL,
        mode="single_turn",
        instruction="""
You answer customer questions for NimbusCloud using ONLY the CONTEXT in the message.
- Cite every fact with its chunk id in square brackets, for example [refund-policy#0].
- If the context does not contain the answer, reply exactly:
  I could not find that in the NimbusCloud knowledge base.
- Never use outside knowledge. Keep answers under 80 words.
""",
    )

    # Node 2b: the fallback branch.
    if fallback == "web":
        fallback_node = Agent(
            name="web_fallback",
            model=MODEL,
            mode="single_turn",
            tools=[google_search],
            instruction=("The internal knowledge base had no answer. Use Google Search to answer the "
                         "question briefly. Start with: 'Not in our knowledge base, but from the web:'"),
        )
    else:
        def fallback_node(node_input):
            return Event(message="I could not find that in the NimbusCloud knowledge base. "
                                 "Please contact support@nimbuscloud.example for help.")

    return Workflow(
        name=name,
        edges=[
            (START, retrieve_context),
            (retrieve_context, {"ANSWER": grounded_answerer, "NO_CONTEXT": fallback_node}),
        ],
    )


kb_rag = make_rag_workflow()
print("RAG workflow ready.")

RAG workflow ready.


In [31]:
# ------------------------------------------------------------------
# Ask questions: two answerable from the KB, one outside the KB.
# ------------------------------------------------------------------
questions = [
    "I bought an annual plan 10 days ago. Can I get a refund, and how long will it take?",
    "What uptime does the Enterprise plan guarantee and what happens if you miss it?",
    "What is the capital of Australia?",
]

for i, q in enumerate(questions):
    await run(kb_rag, q, session_id=f"rag_{i}")

USER > I bought an annual plan 10 days ago. Can I get a refund, and how long will it take?
--------------------------------------------------------------------------------
  [retriever] 3 relevant chunk(s): ['refund-policy#0', 'refund-policy#1', 'pricing-plans#1']
[grounded_answerer]: Yes, you can get a full refund since NimbusCloud offers refunds for annual plans within 14 days of purchase [refund-policy#0]. It will take 5 to 7 business days for the refund to be processed back to your original payment method [refund-policy#0][refund-policy#1].

USER > What uptime does the Enterprise plan guarantee and what happens if you miss it?
--------------------------------------------------------------------------------
  [retriever] 3 relevant chunk(s): ['sla-uptime#0', 'pricing-plans#0', 'pricing-plans#1']
[grounded_answerer]: The Enterprise plan guarantees a 99.99 percent monthly uptime [sla-uptime#0]. If uptime falls below this target, customers receive service credits equal to 10 percent of

## 3.4 Practical: Google Search grounding

For **public and fresh** information, ADK provides the built-in `google_search` tool. Gemini decides when to search, reads results, and answers with grounding metadata (source titles and URLs), which our helper prints with `show_sources=True`.

> **Notes**
> - Built-in tools such as `google_search` are best placed on a dedicated agent rather than mixed with many custom function tools.
> - Grounding with Google Search is billed to your Google Cloud project per grounded request, in addition to model tokens.
> - If you ship search grounded answers in a product, review Google's display requirements for grounding with Google Search (for example showing search suggestions).

In [32]:
# ------------------------------------------------------------------
# A research agent grounded in live Google Search results.
# ------------------------------------------------------------------
web_researcher = Agent(
    name="web_researcher",
    model=MODEL,
    tools=[google_search],
    instruction="""
You are a careful research assistant.
- Use Google Search for anything time-sensitive or factual.
- Answer in 3 to 5 bullet points.
- Mention the date or recency of the information when relevant.
""",
)

await run(web_researcher, "What are the most recent stable releases of Python and what's new in them?",
          session_id="web_1", show_sources=True)

USER > What are the most recent stable releases of Python and what's new in them?
--------------------------------------------------------------------------------
[web_researcher]: As of September 2026, the most recent stable releases of Python are **Python 3.14** and **Python 3.13**, with Python 3.15 scheduled to debut its first stable release on October 1, 2026. Here are the details of the latest stable versions and what is new in them:

*   **Python 3.14 (Latest Major Stable Release):** Originally launched on October 7, 2025, and most recently updated with the stable bugfix version **3.14.7** on August 5, 2026. Major additions include:
    *   **Deferred Annotation Evaluation (PEP 649):** Type hints are now evaluated lazily (on-demand) rather than during class or function definition, resolving runtime overhead and eliminating forward reference issues.
    *   **Official Free-Threaded Support (PEP 779):** Building on the experiments of 3.13, this version officially supports running P

'As of September 2026, the most recent stable releases of Python are **Python 3.14** and **Python 3.13**, with Python 3.15 scheduled to debut its first stable release on October 1, 2026. Here are the details of the latest stable versions and what is new in them:\n\n*   **Python 3.14 (Latest Major Stable Release):** Originally launched on October 7, 2025, and most recently updated with the stable bugfix version **3.14.7** on August 5, 2026. Major additions include:\n    *   **Deferred Annotation Evaluation (PEP 649):** Type hints are now evaluated lazily (on-demand) rather than during class or function definition, resolving runtime overhead and eliminating forward reference issues.\n    *   **Official Free-Threaded Support (PEP 779):** Building on the experiments of 3.13, this version officially supports running Python with the Global Interpreter Lock (GIL) disabled, unlocking true multi-core CPU concurrency.\n    *   **Template String Literals (PEP 750):** Introduces t-strings (e.g., `

## 3.5 Practical: Hybrid grounding (internal first, web as fallback)

Same graph, different fallback: questions covered by the knowledge base use RAG with citations; anything else is answered with Google Search instead of a refusal.

In [33]:
# ------------------------------------------------------------------
# Reuse the factory with fallback="web".
# ------------------------------------------------------------------
hybrid_rag = make_rag_workflow(name="hybrid_rag", fallback="web")

await run(hybrid_rag, "Which security certifications does NimbusCloud have?", session_id="hybrid_1")
await run(hybrid_rag, "What are the main obligations the EU AI Act places on companies?", session_id="hybrid_2",
          show_sources=True)

USER > Which security certifications does NimbusCloud have?
--------------------------------------------------------------------------------
  [retriever] 2 relevant chunk(s): ['security#1', 'security#0']
[grounded_answerer]: NimbusCloud has SOC 2 Type II and ISO 27001 certifications [security#1].

USER > What are the main obligations the EU AI Act places on companies?
--------------------------------------------------------------------------------
  [retriever] 0 relevant chunk(s): []
[web_fallback]: Not in our knowledge base, but from the web:

The EU AI Act places obligations on companies depending on their role (e.g., **providers/developers** versus **deployers/users**) and the classified **risk level** of the AI system (unacceptable, high, limited, or minimal risk). The most demanding requirements apply to companies dealing with **high-risk AI systems** and **General-Purpose AI (GPAI) models**. 

The main obligations include:

### 1. Obligations for Providers of High-Risk AI Syste

"Not in our knowledge base, but from the web:\n\nThe EU AI Act places obligations on companies depending on their role (e.g., **providers/developers** versus **deployers/users**) and the classified **risk level** of the AI system (unacceptable, high, limited, or minimal risk). The most demanding requirements apply to companies dealing with **high-risk AI systems** and **General-Purpose AI (GPAI) models**. \n\nThe main obligations include:\n\n### 1. Obligations for Providers of High-Risk AI Systems\nHigh-risk systems (e.g., those used in biometrics, critical infrastructure, employment, education, or law enforcement) must meet strict standards before entering the EU market:\n* **Risk and Quality Management:** Companies must establish a risk management system throughout the AI's lifecycle and implement a Quality Management System (QMS).\n* **Data Governance:** Training, testing, and validation datasets must be high-quality, relevant, sufficiently representative, and as error-free as possi

## Part 3 Recap

- **Grounding** reduces hallucination by supplying trusted facts at request time.
- **RAG** = chunk, index, retrieve, gate, augment, generate with citations.
- Building RAG as a **graph** makes retrieval and the relevance gate deterministic and debuggable.
- **`google_search`** grounds answers in fresh public web data with source metadata.
- **Hybrid grounding** routes between private knowledge and the web.

**Production upgrades**
- Replace TF-IDF with embeddings (for example Gemini embedding models) and a vector store.
- Use managed retrieval such as Vertex AI Search or Vertex AI RAG Engine for large corpora.
- Add a re-ranking step and log `rag_sources` from state for evaluation.

**Try it yourself**
1. Add a document about "API rate limits" and ask a question about it.
2. Lower `RELEVANCE_THRESHOLD` to `0.01` and observe what happens with unrelated questions.
3. Add a node after `grounded_answerer` that checks the answer contains at least one `[citation]` and routes back if not.

---
# Part 4: Agentic Patterns (ADK Standard Approach)

## 4.1 Concepts

An **agentic system** does more than answer: it takes actions, checks its own work, and plans. Most production agents combine a handful of well known patterns:

| Pattern | Idea | ADK standard implementation |
|---|---|---|
| **Tool use (ReAct)** | Reason, call a tool, observe, repeat | `Agent(tools=[python_functions])`; ADK runs the tool loop |
| **Reflection** | Draft, critique, revise until good | Graph loop: writer agent, critic agent with `output_schema`, gate function with max iterations |
| **Planning** | Break a goal into steps, then execute | Planner agent with `output_schema`, then executor agent with tools |
| **Routing** | Send work to the right handler | Router node plus route edges (Part 2) |
| **Parallelization** | Do independent work at the same time | Fan-out plus `JoinNode` (Part 2) |
| **Multi-agent** | Specialists collaborate | Coordinator with `sub_agents` (Part 5) |

### The ADK rule of thumb

> **Let the LLM decide what needs judgment. Let code decide what must be guaranteed.**

- Which tool to call, how to phrase an answer: **agent** decides.
- Maximum number of retries, approval thresholds, output validation: **graph and functions** decide.

### How ADK function tools work

You write a normal Python function with **type hints and a docstring**. ADK turns the signature and docstring into a tool declaration for Gemini. When Gemini asks for the tool, ADK executes it and feeds the result back automatically.

```
User -> Agent -> Gemini: "call get_order_status(order_id='ORD-1002')"
              -> ADK runs your Python function -> result dict -> Gemini -> final answer
```

**Best practices for tools:** clear names, a docstring that explains when to use it, simple typed parameters, and return a `dict` that includes a `status` field so the model can handle errors gracefully.

## 4.2 Practical: Define tools

In [34]:
# ------------------------------------------------------------------
# Tool 1: safe calculator (never use eval() on model generated text!).
# ------------------------------------------------------------------
_ALLOWED_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
                ast.Div: operator.truediv, ast.Pow: operator.pow, ast.USub: operator.neg,
                ast.Mod: operator.mod}


def _safe_eval(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _ALLOWED_OPS:
        return _ALLOWED_OPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _ALLOWED_OPS:
        return _ALLOWED_OPS[type(node.op)](_safe_eval(node.operand))
    raise ValueError("Unsupported expression")


def calculator(expression: str) -> dict:
    """Evaluate a basic arithmetic expression such as '(250 * 1.18) + 15'.

    Use this for ANY math instead of calculating in your head.

    Args:
        expression: Arithmetic using numbers and + - * / ** % and parentheses.
    """
    try:
        result = _safe_eval(ast.parse(expression, mode="eval").body)
        return {"status": "success", "expression": expression, "result": round(result, 4)}
    except Exception as exc:
        return {"status": "error", "message": f"Could not evaluate: {exc}"}


# ------------------------------------------------------------------
# Tool 2: currency conversion (mock rates for a reproducible tutorial).
# ------------------------------------------------------------------
_RATES_TO_USD = {"USD": 1.0, "EUR": 1.08, "GBP": 1.27, "INR": 0.012, "JPY": 0.0067}


def convert_currency(amount: float, from_currency: str, to_currency: str) -> dict:
    """Convert an amount between currencies (USD, EUR, GBP, INR, JPY).

    Args:
        amount: The amount of money to convert.
        from_currency: Three-letter source currency code, e.g. 'EUR'.
        to_currency: Three-letter target currency code, e.g. 'USD'.
    """
    src, dst = from_currency.upper(), to_currency.upper()
    if src not in _RATES_TO_USD or dst not in _RATES_TO_USD:
        return {"status": "error", "message": f"Unsupported currency. Use one of {list(_RATES_TO_USD)}"}
    converted = amount * _RATES_TO_USD[src] / _RATES_TO_USD[dst]
    return {"status": "success", "amount": amount, "from": src, "to": dst, "converted": round(converted, 2)}


# ------------------------------------------------------------------
# Tool 3: order lookup (mock database).
# ------------------------------------------------------------------
_ORDERS = {
    "ORD-1001": {"item": "Wireless Headphones", "price": 120.0, "currency": "USD", "delivery_status": "delivered"},
    "ORD-1002": {"item": "Mechanical Keyboard", "price": 250.0, "currency": "EUR", "delivery_status": "shipped",
                 "eta": "2 business days"},
    "ORD-1003": {"item": "4K Monitor", "price": 38000.0, "currency": "INR", "delivery_status": "processing"},
}


def get_order_status(order_id: str) -> dict:
    """Look up an order's item, price, currency, and delivery status.

    Args:
        order_id: Order identifier in the format ORD-1234.
    """
    order = _ORDERS.get(order_id.strip().upper())
    if not order:
        return {"status": "error", "message": f"Order {order_id} not found."}
    return {"status": "success", "order_id": order_id.upper(), **order}


# Quick local test of the tools (plain Python, no LLM involved).
print(calculator("(250 * 1.18) + 15"))
print(convert_currency(250, "EUR", "USD"))
print(get_order_status("ORD-1002"))

{'status': 'success', 'expression': '(250 * 1.18) + 15', 'result': 310.0}
{'status': 'success', 'amount': 250, 'from': 'EUR', 'to': 'USD', 'converted': 270.0}
{'status': 'success', 'order_id': 'ORD-1002', 'item': 'Mechanical Keyboard', 'price': 250.0, 'currency': 'EUR', 'delivery_status': 'shipped', 'eta': '2 business days'}


## 4.3 Practical: Tool use (ReAct) agent

Gemini reasons about which tools to call, possibly in several steps, and ADK executes them. Watch the `[tool call]` and `[tool result]` lines.

In [35]:
# ------------------------------------------------------------------
# A shopping assistant that must use tools for facts and math.
# ------------------------------------------------------------------
shopping_assistant = Agent(
    name="shopping_assistant",
    model=MODEL,
    tools=[get_order_status, convert_currency, calculator],
    instruction="""
You are a shopping assistant.
- ALWAYS use get_order_status for order facts. Never guess order details.
- ALWAYS use convert_currency for currency conversion and calculator for arithmetic.
- If a tool returns status "error", explain the problem and ask for corrected input.
- Show the final numbers clearly in 2 to 4 short lines.
""",
)

# This requires several tool calls: lookup -> convert -> calculate tax.
await run(shopping_assistant,
          "For order ORD-1002, what is the price in USD, and what's the total if I add 18% import tax?",
          session_id="shop_1")

# Error handling: the order does not exist.
await run(shopping_assistant, "Where is my order ORD-9999?", session_id="shop_2")

USER > For order ORD-1002, what is the price in USD, and what's the total if I add 18% import tax?
--------------------------------------------------------------------------------
  [tool call]   shopping_assistant -> get_order_status({'order_id': 'ORD-1002'})
  [tool result] get_order_status -> {'status': 'success', 'order_id': 'ORD-1002', 'item': 'Mechanical Keyboard', 'price': 250.0, 'currency': 'EUR', 'delivery_status': 'shipped', 'eta': '2 business days'}
  [tool call]   shopping_assistant -> convert_currency({'amount': 250, 'from_currency': 'EUR', 'to_currency': 'USD'})
  [tool result] convert_currency -> {'status': 'success', 'amount': 250, 'from': 'EUR', 'to': 'USD', 'converted': 270.0}
  [tool call]   shopping_assistant -> calculator({'expression': '270 * 1.18'})
  [tool result] calculator -> {'status': 'success', 'expression': '270 * 1.18', 'result': 318.6}
[shopping_assistant]: The details for your order are as follows:

* **Item**: Mechanical Keyboard (ORD-1002)
* **Base Pr

"I'm sorry, but we couldn't find any details for order ORD-9999. \n\nPlease double-check your order number and try again."

## 4.4 Practical: Reflection loop (writer, critic, gate)

```
START -> capture_topic -> writer -> save_draft -> critic (JSON verdict) -> quality_gate
                            ^                                               |      |
                            |____________________ REVISE ___________________|      |
                                                                          APPROVE -> publish
```

- The **critic** returns structured JSON (`output_schema`), so the verdict is machine readable.
- The **quality gate** is plain code: it enforces a **maximum number of revisions** so the loop can never run forever.

In [36]:
MAX_REVISIONS = 3


# Structured critique so code can read the verdict reliably.
class Critique(BaseModel):
    verdict: Literal["APPROVE", "REVISE"]
    score: int = Field(description="Quality score from 1 to 10")
    feedback: str = Field(description="Specific, actionable improvement advice")


# Node 1: store the brief and reset the attempt counter.
def capture_topic(node_input, ctx):
    ctx.state["brief"] = to_text(node_input).strip()
    ctx.state["revision"] = 0
    return f"Write the first draft for this brief: {ctx.state['brief']}"


# Node 2 (agent): writes or rewrites the draft.
writer = Agent(
    name="writer", model=MODEL, mode="single_turn",
    instruction=("You are a marketing copywriter. Write a product announcement of at most 60 words. "
                 "If feedback is provided, apply ALL of it. Output only the announcement text."),
)


# Node 3: remember the latest draft, then ask the critic to review it.
def save_draft(node_input, ctx):
    ctx.state["draft"] = to_text(node_input).strip()
    return f"BRIEF: {ctx.state['brief']}\n\nDRAFT:\n{ctx.state['draft']}"


# Node 4 (agent): strict reviewer with a JSON output schema.
critic = Agent(
    name="critic", model=MODEL, mode="single_turn", output_schema=Critique,
    instruction="""
You are a demanding editor. Review the DRAFT against the BRIEF.
APPROVE only if ALL are true: at most 60 words, mentions a concrete benefit, has a clear call to action,
and contains no vague hype words such as "revolutionary" or "game-changing".
Otherwise return REVISE with specific feedback.
""",
)


# Node 5 (function): the deterministic loop controller.
def quality_gate(node_input, ctx):
    review = to_dict(node_input)
    ctx.state["revision"] += 1
    n = ctx.state["revision"]
    print(f"  [gate] round {n}: verdict={review.get('verdict')} score={review.get('score')}")

    if review.get("verdict") == "APPROVE" or n >= MAX_REVISIONS:
        return Event(output=ctx.state["draft"], route="APPROVE")

    revision_prompt = (f"BRIEF: {ctx.state['brief']}\n\nPREVIOUS DRAFT:\n{ctx.state['draft']}\n\n"
                       f"EDITOR FEEDBACK: {review.get('feedback')}\n\nWrite an improved version.")
    return Event(output=revision_prompt, route="REVISE")


# Node 6: final message to the user.
def publish(node_input, ctx):
    return Event(message=f"FINAL COPY (after {ctx.state['revision']} review round(s)):\n\n{to_text(node_input)}")


reflection_loop = Workflow(
    name="reflection_loop",
    edges=[
        (START, capture_topic, writer, save_draft, critic, quality_gate),
        (quality_gate, {"REVISE": writer, "APPROVE": publish}),   # REVISE edge creates the loop
    ],
)

await run(reflection_loop,
          "Announce SnapNotes 2.0: a note-taking app that now transcribes voice memos offline.",
          session_id="reflect_1")

USER > Announce SnapNotes 2.0: a note-taking app that now transcribes voice memos offline.
--------------------------------------------------------------------------------
[writer]: Meet SnapNotes 2.0, the ultimate note-taking app! We’ve leveled up with a game-changing new feature: offline voice memo transcription. Now you can capture your thoughts and convert speech to text instantly, anywhere—no internet required. Keep your ideas flowing and your data private. Download SnapNotes 2.0 today and never miss a beat!

[critic]: {"verdict": "REVISE", "score": 6, "feedback": "Your draft exceeds limits on hype words by using 'game-changing' and 'ultimate'. Remove these cliché marketing buzzwords. Keep the concrete benefit (offline transcription/privacy) and the clear call to action, but express them with simpler, more direct language."}

  [gate] round 1: verdict=REVISE score=6
[writer]: Meet SnapNotes 2.0. Now you can transcribe voice memos to text directly on your device—no internet connect

'FINAL COPY (after 2 review round(s)):\n\nMeet SnapNotes 2.0. Now you can transcribe voice memos to text directly on your device—no internet connection required. Keep your ideas organized and your data completely private, wherever you go. Download SnapNotes 2.0 today to start transcribing.'

## 4.5 Practical: Plan-and-execute

For multi-step requests, a **planner** first produces a structured plan, then an **executor** with tools carries it out. Separating the two makes the reasoning visible and easier to audit.

In [37]:
# ------------------------------------------------------------------
# Plan schema: an ordered list of steps with a suggested tool.
# ------------------------------------------------------------------
class PlanStep(BaseModel):
    step: int
    action: str = Field(description="What to do in this step")
    tool: Literal["get_order_status", "convert_currency", "calculator", "none"]


class Plan(BaseModel):
    goal: str
    steps: list[PlanStep]


def capture_request(node_input, ctx):
    ctx.state["request"] = to_text(node_input).strip()
    return ctx.state["request"]


planner = Agent(
    name="planner", model=MODEL, mode="single_turn", output_schema=Plan,
    instruction=("Create a short step-by-step plan (max 6 steps) to fulfil the user's request. "
                 "Available tools: get_order_status, convert_currency, calculator. Do not execute anything."),
)


# Show the plan to the user, then hand request + plan to the executor.
def format_plan(node_input, ctx):
    plan = to_dict(node_input)
    lines = [f"  {s['step']}. {s['action']}  (tool: {s['tool']})" for s in plan.get("steps", [])]
    print("  [plan]\n" + "\n".join(lines))
    return f"REQUEST: {ctx.state['request']}\n\nPLAN:\n" + "\n".join(lines)


executor = Agent(
    name="executor", model=MODEL, mode="single_turn",
    tools=[get_order_status, convert_currency, calculator],
    instruction=("Execute the PLAN step by step using the tools. Do not skip steps. "
                 "Finish with a short summary table of the results and a one-line answer."),
)

plan_and_execute = Workflow(
    name="plan_and_execute",
    edges=[(START, capture_request, planner, format_plan, executor)],
)

await run(plan_and_execute,
          "Compare the prices of orders ORD-1001, ORD-1002 and ORD-1003 in USD and tell me the total.",
          session_id="plan_1")

USER > Compare the prices of orders ORD-1001, ORD-1002 and ORD-1003 in USD and tell me the total.
--------------------------------------------------------------------------------
[planner]: {"goal": "Compare the prices of ORD-1001, ORD-1002, and ORD-1003 in USD and calculate their total sum.", "steps": [{"step": 1, "action": "Retrieve the details and price of order ORD-1001", "tool": "get_order_status"}, {"step": 2, "action": "Retrieve the details and price of order ORD-1002", "tool": "get_order_status"}, {"step": 3, "action": "Retrieve the details and price of order ORD-1003", "tool": "get_order_status"}, {"step": 4, "action": "Convert the prices of the three orders into USD", "tool": "convert_currency"}, {"step": 5, "action": "Calculate the total sum of the three USD prices", "tool": "calculator"}]}

  [plan]
  1. Retrieve the details and price of order ORD-1001  (tool: get_order_status)
  2. Retrieve the details and price of order ORD-1002  (tool: get_order_status)
  3. Retrieve the

'| Order ID | Original Price | Converted Price (USD) | Status |\n| :--- | :--- | :--- | :--- |\n| **ORD-1001** | 120 USD | $120.00 | Delivered |\n| **ORD-1002** | 250 EUR | $270.00 | Shipped |\n| **ORD-1003** | 38,000 INR | $456.00 | Processing |\n| **Total** | | **$846.00** | |\n\nThe total price of the three orders is **$846.00 USD**.'

## Part 4 Recap

| Pattern | Who decides | Guardrail in code |
|---|---|---|
| Tool use | Agent picks tools | Tools validate input, return `status` |
| Reflection | Critic judges quality | Gate enforces `MAX_REVISIONS` |
| Plan-and-execute | Planner designs steps | Schema limits tools and step count |

**Try it yourself**
1. Add a `get_shipping_quote(country: str, weight_kg: float)` tool and ask the shopping assistant a shipping question.
2. Make the critic stricter (for example "must include an emoji") and watch the loop run more rounds.
3. Add a human-approval style step: route `APPROVE` to a node that stores the draft in state and asks the user to reply "publish".

---
# Part 5: Multi-Agent Solution

## 5.1 Concepts

### Why multiple agents?

One agent with 30 tools and a 3 page instruction becomes slow, expensive, and unreliable. Splitting work into **specialists** gives each agent a focused instruction and a small tool set, which improves accuracy and makes testing easier.

### Three architectures in ADK 2.0

| Architecture | Who controls flow | Best when | ADK building blocks |
|---|---|---|---|
| **Coordinator (dispatcher)** | An LLM coordinator | The user's request decides which expert is needed | `Agent(sub_agents=[...])`, specialists with `description` and `mode` |
| **Graph team** | Your edges | The process is known in advance | `Workflow`, agent nodes, `JoinNode` |
| **Hybrid** | Graph for the backbone, LLMs at decision points | Real production systems | Workflow + router + specialist agents + nested workflows |

```
Coordinator                     Graph team (parallel panel)          Hybrid (capstone)
-----------                     ---------------------------          -----------------
        coordinator             START -> market_analyst -\           START -> intake -> triage(agent)
       /     |      \           START -> tech_analyst   --> Join     -> dispatch (business rules)
 order   finance   policy       START -> risk_analyst   -/   |          |-> POLICY    -> RAG sub-workflow
 agent    agent    agent                         lead_strategist        |-> ORDER     -> order agent
                                                                        |-> TECHNICAL -> tech agent
                                                                        |-> ESCALATE  -> human handoff
```

### Key ideas for collaboration

- **`description` is the contract.** The coordinator reads specialist descriptions to choose who to call, so make them specific and non-overlapping.
- **`mode="single_turn"` specialists** do one job and return the result to the coordinator automatically.
- ADK auto-creates a **delegation tool** named after each sub-agent.
- **Context isolation:** single-turn and task agents work in their own session branch, so parallel specialists do not see each other's intermediate work.

## 5.2 Practical: Coordinator with specialist sub-agents

We reuse the Part 4 tools and the Part 3 retriever. Wrapping the retriever as a **tool** is called *agentic RAG*: the agent decides when to search.

In [38]:
# ------------------------------------------------------------------
# Agentic RAG tool: wraps the Part 3 retriever so an agent can search on demand.
# ------------------------------------------------------------------
def search_knowledge_base(query: str) -> dict:
    """Search NimbusCloud's policy knowledge base (refunds, pricing, SLA, security, support, data retention).

    Args:
        query: A focused search query, e.g. 'annual plan refund window'.
    """
    hits = [h for h in retriever.search(query, k=3) if h["score"] >= RELEVANCE_THRESHOLD]
    if not hits:
        return {"status": "no_results", "message": "Nothing relevant found in the knowledge base."}
    return {"status": "success",
            "results": [{"chunk_id": h["chunk_id"], "text": h["text"]} for h in hits]}


# ------------------------------------------------------------------
# Specialists: focused instruction, small tool set, clear description, single_turn mode.
# ------------------------------------------------------------------
order_specialist = Agent(
    name="order_specialist", model=MODEL, mode="single_turn",
    description="Looks up order status, items, prices, and delivery ETAs by order id.",
    tools=[get_order_status],
    instruction="Use get_order_status to answer order questions. Report facts only, in 1 to 3 lines.",
)

finance_specialist = Agent(
    name="finance_specialist", model=MODEL, mode="single_turn",
    description="Performs currency conversions and price, tax, or discount calculations.",
    tools=[convert_currency, calculator],
    instruction="Use the tools for every number. Show the calculation result concisely.",
)

policy_specialist = Agent(
    name="policy_specialist", model=MODEL, mode="single_turn",
    description="Answers questions about NimbusCloud policies: refunds, pricing plans, SLA, security, support, data retention.",
    tools=[search_knowledge_base],
    instruction="Search the knowledge base, answer only from results, and cite chunk ids in brackets.",
)

# ------------------------------------------------------------------
# The coordinator: no tools of its own, just delegation and final synthesis.
# ------------------------------------------------------------------
support_coordinator = Agent(
    name="support_coordinator",
    model=MODEL,
    sub_agents=[order_specialist, finance_specialist, policy_specialist],
    instruction="""
You are the front-desk coordinator for a customer support team.
1. Break the customer's request into parts.
2. Delegate each part to the most suitable specialist. You may call several specialists in sequence
   and pass results from one to the next (for example an order price to the finance specialist).
3. Do NOT answer facts yourself. Combine the specialists' results into one friendly reply under 120 words.
""",
)
print("Coordinator team ready:", [a.name for a in support_coordinator.sub_agents])

Coordinator team ready: ['order_specialist', 'finance_specialist', 'policy_specialist']


In [39]:
# ------------------------------------------------------------------
# Single-domain and multi-domain requests. Watch which specialists get called.
# ------------------------------------------------------------------
await run(support_coordinator, "What's the status of order ORD-1003?", session_id="coord_1")

await run(support_coordinator,
          "My order ORD-1002 was in EUR. What is that in GBP, and what's your refund policy on annual plans?",
          session_id="coord_2")

USER > What's the status of order ORD-1003?
--------------------------------------------------------------------------------
  [tool call]   support_coordinator -> order_specialist({'request': 'What is the status of order ORD-1003?'})
  [tool call]   order_specialist -> get_order_status({'order_id': 'ORD-1003'})
  [tool result] get_order_status -> {'status': 'success', 'order_id': 'ORD-1003', 'item': '4K Monitor', 'price': 38000.0, 'currency': 'INR', 'delivery_status': 'processing'}
[order_specialist]: Order ORD-1003 is for a 4K Monitor costing 38,000 INR. 
The delivery status is currently processing.

  [tool result] order_specialist -> {'result': 'Order ORD-1003 is for a 4K Monitor costing 38,000 INR. \nThe delivery status is currently processing.'}
[support_coordinator]: Your order ORD-1003 for a 4K Monitor (valued at 38,000 INR) is currently processing. We will update you as soon as it ships!

USER > My order ORD-1002 was in EUR. What is that in GBP, and what's your refund policy o

'Your order ORD-1002 total of 250 EUR is approximately 212.60 GBP. \n\nRegarding our refund policy, NimbusCloud offers a full refund on annual plans if requested within 14 days of purchase. Once processed, your refund will be returned to your original payment method within 5 to 7 business days. Please note that enterprise contracts follow the specific terms outlined in their signed agreements. \n\nLet me know if you need help with anything else!'

## 5.3 Practical: Parallel expert panel (graph team)

When you always want **all** perspectives, a graph is cheaper and more predictable than asking a coordinator to remember to call everyone. Three analysts run in parallel; a lead strategist synthesizes a structured decision.

In [40]:
# ------------------------------------------------------------------
# Three parallel analysts. Each sees the same user proposal (output of START).
# ------------------------------------------------------------------
market_analyst = Agent(name="market_analyst", model=MODEL, mode="single_turn",
    instruction="Assess market demand, target customers, and competitors in 4 bullet points.")
tech_analyst = Agent(name="tech_analyst", model=MODEL, mode="single_turn",
    instruction="Assess technical feasibility, build complexity, and key dependencies in 4 bullet points.")
risk_analyst = Agent(name="risk_analyst", model=MODEL, mode="single_turn",
    instruction="Identify legal, financial, and reputational risks with mitigations in 4 bullet points.")

panel_join = JoinNode(name="panel_join")


# Merge the three reports into one prompt, labelled by analyst name.
def compile_reports(node_input):
    sections = [f"### {name.upper()}\n{to_text(report)}" for name, report in node_input.items()]
    return "Analyst reports:\n\n" + "\n\n".join(sections)


class LaunchDecision(BaseModel):
    decision: Literal["GO", "NO_GO", "PILOT_FIRST"]
    confidence: int = Field(description="Confidence from 1 to 10")
    top_reasons: list[str] = Field(description="Three most important reasons")
    next_steps: list[str] = Field(description="Three concrete next steps")


lead_strategist = Agent(name="lead_strategist", model=MODEL, mode="single_turn",
    output_schema=LaunchDecision,
    instruction="You are the head of strategy. Weigh all analyst reports fairly and make a launch decision.")


def present_decision(node_input):
    d = to_dict(node_input)
    return Event(message=(
        f"DECISION: {d.get('decision')}  (confidence {d.get('confidence')}/10)\n\n"
        "Reasons:\n" + "\n".join(f"  - {r}" for r in d.get("top_reasons", [])) +
        "\n\nNext steps:\n" + "\n".join(f"  - {s}" for s in d.get("next_steps", []))
    ))


expert_panel = Workflow(
    name="expert_panel",
    edges=[
        (START, market_analyst, panel_join),
        (START, tech_analyst, panel_join),
        (START, risk_analyst, panel_join),
        (panel_join, compile_reports, lead_strategist, present_decision),
    ],
)

await run(expert_panel,
          "Proposal: launch an AI meal-planning feature inside our fitness app for paying subscribers.",
          session_id="panel_1")

USER > Proposal: launch an AI meal-planning feature inside our fitness app for paying subscribers.
--------------------------------------------------------------------------------
[market_analyst]: * **Market Demand:** There is high and growing demand for "all-in-one" wellness platforms, as modern consumers increasingly seek integrated, AI-driven solutions that seamlessly bridge the gap between workout tracking and personalized nutrition to save time and effort.
* **Target Customers (Goal-Oriented):** Existing premium subscribers who are highly motivated to achieve specific fitness outcomes (e.g., weight loss, muscle gain) and want automated, dynamic calorie/macro-tracking tied directly to their daily exercise.
* **Target Customers (Convenience-Driven):** Busy, health-conscious professionals who suffer from meal-prep decision fatigue and value instant, AI-generated grocery lists and recipes tailored to their specific dietary restrictions (e.g., vegan, keto, gluten-free).
* **Competitor

'DECISION: PILOT_FIRST  (confidence 8/10)\n\nReasons:\n  - High market demand for integrated, AI-driven wellness solutions among premium, goal-oriented subscribers.\n  - Technical feasibility is high leveraging modern LLMs, though moderate complexity exists in grounding data with verified food databases.\n  - Critical legal, financial, and reputational risks regarding medical liability, data privacy, and API costs must be validated in a controlled environment.\n\nNext steps:\n  - Launch a limited pilot program for a subset of premium subscribers to test the AI meal planner and gather user feedback.\n  - Build and integrate the verification layer with USDA/Edamam APIs and implement hard-coded safety exclusions and medical disclaimers.\n  - Secure zero-data-retention enterprise agreements with LLM providers and set up caching/rate-limiting to control API costs.'

## 5.4 Capstone: Hybrid customer support system

This final system combines **everything** from the tutorial:

| Tutorial part | Used here as |
|---|---|
| Part 1 structured output | `triage_agent` returns a typed `Ticket` |
| Part 2 routing and state | `dispatch` applies business rules and stores the ticket in state |
| Part 3 RAG | A **nested** RAG workflow answers policy questions with citations |
| Part 4 tool use | Order and technical agents use tools |
| Part 5 multi-agent | Several specialists behind one entry point |

**Business rule enforced in code (not in a prompt):** any `high` urgency ticket is escalated to a human, regardless of category.

**Retrieval tip:** the triage agent rewrites messy customer text into a clean standalone `question`. That rewritten question is what the RAG sub-workflow searches with, which noticeably improves keyword and embedding retrieval.

In [41]:
# ------------------------------------------------------------------
# Extra tool for the technical branch (mock status page).
# ------------------------------------------------------------------
def get_service_status(service_name: str) -> dict:
    """Check the live status of a NimbusCloud service: sync, web-app, api, or mobile-app.

    Args:
        service_name: One of sync, web-app, api, mobile-app.
    """
    statuses = {"sync": "degraded: delayed file sync in EU region, fix in progress",
                "web-app": "operational", "api": "operational", "mobile-app": "operational"}
    key = service_name.strip().lower()
    return {"status": "success", "service": key, "state": statuses.get(key, "unknown service")}


# ------------------------------------------------------------------
# Triage schema: the LLM extracts structure; code makes the routing decision.
# ------------------------------------------------------------------
class Ticket(BaseModel):
    category: Literal["POLICY", "ORDER", "TECHNICAL", "OTHER"]
    urgency: Literal["low", "medium", "high"]
    summary: str = Field(description="One sentence summary of the issue")
    question: str = Field(description="The customer's request rewritten as a clear standalone question")


def intake(node_input, ctx):
    ctx.state["raw_message"] = to_text(node_input).strip()
    return ctx.state["raw_message"]


triage_agent = Agent(
    name="triage_agent", model=MODEL, mode="single_turn", output_schema=Ticket,
    instruction="""
Triage the customer message.
POLICY = refunds, pricing, plans, SLA, security, data retention, support hours.
ORDER = a specific order id or delivery.  TECHNICAL = bugs, outages, errors, sync problems.
Urgency is high ONLY for data loss, security incidents, legal threats, or a business-wide outage.
""",
)


# Deterministic dispatcher: business rules live here, not in prompts.
def dispatch(node_input, ctx):
    ticket = to_dict(node_input)
    ctx.state["ticket"] = ticket
    category, urgency = ticket.get("category", "OTHER"), ticket.get("urgency", "low")
    route = "ESCALATE" if urgency == "high" or category == "OTHER" else category
    print(f"  [dispatch] category={category} urgency={urgency} -> {route}")
    return Event(output=ticket.get("question", ctx.state["raw_message"]), route=route)


capstone_order_agent = Agent(
    name="capstone_order_agent", model=MODEL, mode="single_turn", tools=[get_order_status, convert_currency],
    instruction="Resolve the order question using tools. Reply in under 60 words, friendly and factual.",
)

capstone_tech_agent = Agent(
    name="capstone_tech_agent", model=MODEL, mode="single_turn", tools=[get_service_status],
    instruction=("Check the relevant service status first. If there is a known incident, say so and give a "
                 "workaround. Otherwise give 3 troubleshooting steps. Under 80 words."),
)


def human_handoff(node_input, ctx):
    t = ctx.state["ticket"]
    return Event(message=(
        "ESCALATED TO A HUMAN AGENT\n"
        f"  Summary : {t.get('summary')}\n"
        f"  Urgency : {t.get('urgency')}\n"
        "A senior support engineer will contact you within 1 hour. Reference: TCK-"
        f"{abs(hash(t.get('summary', ''))) % 100000:05d}"
    ))


support_system = Workflow(
    name="support_system",
    edges=[
        (START, intake, triage_agent, dispatch),
        (dispatch, {
            "POLICY": make_rag_workflow(name="policy_rag"),   # nested RAG workflow from Part 3
            "ORDER": capstone_order_agent,
            "TECHNICAL": capstone_tech_agent,
            "ESCALATE": human_handoff,
        }),
    ],
)
print("Capstone support system ready.")

Capstone support system ready.


In [42]:
# ------------------------------------------------------------------
# End-to-end tests: each message should take a different path.
# ------------------------------------------------------------------
capstone_tests = [
    "hi, what is the refund window if I cancel my yearly plan?",
    "Where's my keyboard? Order ORD-1002. Also how much was it in USD?",
    "My files haven't synced since this morning, I'm based in Berlin.",
    "URGENT: someone logged into our admin account from another country and deleted projects!",
]

for i, msg in enumerate(capstone_tests):
    await run(support_system, msg, session_id=f"capstone_{i}")

USER > hi, what is the refund window if I cancel my yearly plan?
--------------------------------------------------------------------------------
[triage_agent]: {"category":"POLICY","urgency":"low","summary":"The customer is inquiring about the refund policy window for canceling a yearly subscription plan.","question":"What is the refund window for canceling a yearly subscription plan?"}

  [dispatch] category=POLICY urgency=low -> POLICY
  [retriever] 3 relevant chunk(s): ['refund-policy#1', 'pricing-plans#0', 'refund-policy#0']
[grounded_answerer]: For annual plans, NimbusCloud offers a full refund within 14 days of purchase [refund-policy#0].

USER > Where's my keyboard? Order ORD-1002. Also how much was it in USD?
--------------------------------------------------------------------------------
[triage_agent]: {"category":"ORDER","urgency":"low","summary":"The customer is inquiring about the delivery status and USD price of their keyboard order, ORD-1002.","question":"What is the d

## Part 5 Recap

- **Coordinator pattern**: flexible, LLM-driven delegation based on specialist `description`s.
- **Graph team**: predictable, parallel, and cheaper when every expert must always contribute.
- **Hybrid**: LLMs extract meaning (triage), code enforces rules (dispatch), specialists and nested workflows do the work.

**Try it yourself**
1. Add a `billing_specialist` sub-agent to `support_coordinator` with a mock `get_invoice(invoice_id)` tool.
2. Add a fourth analyst (for example `ux_analyst`) to `expert_panel`. Only two lines of edges change!
3. In the capstone, route `TECHNICAL` tickets for the `sync` service to `ESCALATE` when more than 2 customers report it (hint: count in `app:` scoped state).

---
# Wrap-up

## What you built

| Part | Skill | Key API |
|---|---|---|
| 1 | Prompt based assistants | `Agent`, `instruction`, `{state}` templates, `output_schema`, `generate_content_config` |
| 2 | Graph workflows | `Workflow`, `START`, function nodes, `Event(output, route, message)`, `ctx.state`, `JoinNode` |
| 3 | RAG and grounding | Retriever + relevance gate route, citations, `google_search`, hybrid fallback |
| 4 | Agentic patterns | Function tools, reflection loop with bounded retries, plan-and-execute |
| 5 | Multi-agent systems | `sub_agents` + `mode="single_turn"`, parallel panels, nested workflows in a hybrid system |

## Best practices checklist

- Put **process** in the graph and **judgment** in agents.
- Use `mode="single_turn"` for agents inside workflows.
- **Validate LLM output** before using it for routing (allowed labels, schemas, fallbacks).
- **Bound every loop** with a counter in state.
- Keep state **small** (ids, flags, counters). Pass large data through `output`, artifacts, or databases.
- Write specific, non-overlapping `description`s for sub-agents.
- Tools: typed parameters, helpful docstrings, and a `status` field in the returned dict.
- Test retrieval and tools **without** the LLM first, then add agents.

## Troubleshooting

| Symptom | Likely cause and fix |
|---|---|
| `ImportError: cannot import name 'Workflow'` | ADK 1.x is loaded. Upgrade to `google-adk>=2.0.0` and restart the kernel |
| `DefaultCredentialsError` | No ADC credentials. Run `gcloud auth application-default login`, then restart the kernel |
| `PERMISSION_DENIED` or `403` | Enable billing, run `gcloud services enable aiplatform.googleapis.com`, and grant your account the Vertex AI User role (`roles/aiplatform.user`) |
| `SERVICE_DISABLED` | The Vertex AI API is off for the project. Enable it with the command above and wait a minute |
| `404 NOT_FOUND` for the model | Wrong project or region in `.env`, or `gemini-3.5-flash` is not enabled in `us-central1` for your project |
| `DeprecationWarning: GOOGLE_GENAI_USE_VERTEXAI` | Harmless. Newer ADK versions print it; everything still works |
| Quota or project warnings from ADC | Run `gcloud auth application-default set-quota-project YOUR_PROJECT_ID` |
| Settings changed in `.env` are not used | Click **Restart** and run the setup cells again |
| `KeyError` when an instruction runs | A `{placeholder}` in the instruction has no matching state key. Set the state or use `{key?}` |
| Workflow stops after the router | The emitted `route` matched no edge key. Normalize labels and add a fallback route |
| `JoinNode` never finishes | One branch produced no output or failed. Make every branch return a value |
| Loop runs too many times | Missing or incorrect exit condition. Enforce a max counter in the gate node |
| `asyncio.run() cannot be called from a running event loop` | In Jupyter use `await` directly instead of `asyncio.run()` |

## Next steps

1. **Visual debugging:** move an agent into a project folder with `root_agent = ...` in `agent.py` and run `adk web` to inspect events and graphs in the browser.
2. **Human-in-the-loop and dynamic workflows:** explore pausing graphs for approval and `@node` functions that call `ctx.run_node(...)`.
3. **Evaluation:** create test sets and run ADK evaluation to measure answer quality and tool usage.
4. **Deployment:** deploy to Cloud Run or the Agent Platform managed runtime (formerly Vertex AI Agent Engine) with persistent session services. Your project based setup carries over unchanged.

## References

- ADK graph workflows: https://adk.dev/graphs/
- Graph routes (sequence, routing, fan-out, loops): https://adk.dev/graphs/routes/
- Data handling (output, message, state, schemas): https://adk.dev/graphs/data-handling/
- Collaborative agents and modes: https://adk.dev/workflows/collaboration/
- Grounding in ADK: https://adk.dev/grounding/
- Gemini 3.5 Flash model page: https://ai.google.dev/gemini-api/docs/models/gemini-3.5-flash

Happy building!